# Trilha B - PU classico sob SCAR (Elkan & Noto)

Este caderno organiza a Trilha B como resposta operacional para a pergunta central: **como corrigimos o vies de selecao assumindo SCAR?**

Papel editorial da trilha:
- **B1**: Elkan-Noto + Logistic Regression como referencia metodologica principal e ancora interpretativa linear;
- **B2**: sensibilidade ao estimador base nao linear, mantendo o mesmo arcabouco SCAR;
- **B3**: calibracao monotona do melhor modelo bruto, com papel interpretativo e nao substitutivo;
- **SHAP de B2**: interpretabilidade nao linear do componente preditivo que sustenta o ranking da trilha;
- **blocos adicionais de robustez**: estabilidade do ranking, sensibilidade a `hold_out_ratio` e diagnostico de `score_pu_raw > 1`.

Definicoes conceituais centrais:
- `s` e o rotulo observado de selecao: beneficiario observado do credito (`s=1`) versus caso unlabeled (`s=0`);
- `y` e o alvo latente de interesse, nao plenamente observado;
- SCAR assume que a chance de um positivo real aparecer como positivo rotulado e aproximadamente constante;
- Elkan-Noto treina um classificador para o problema observado em `s` e corrige o score pela constante `c = P(s=1 | y=1)`.

Produto principal da trilha:
- score bruto oficial para ranking operacional, score calibrado para apoio interpretativo, metricas proxy em `s`, diagnosticos de score, interpretabilidade e comparacao interna das variantes do mesmo arcabouco PU classico sob SCAR.

O que a trilha **nao** demonstra:
- nao recupera `y` verdadeiro observado;
- nao valida causalmente a elegibilidade real;
- nao elimina a limitacao estrutural da hipotese SCAR;
- nao autoriza leitura ingenua dos scores como probabilidades causais perfeitas.

Ponte conceitual entre trilhas:
- **Trilha A**: o que aprendemos diretamente de `P(s|x)` no baseline inicial;
- **Trilha B**: como corrigimos o vies de selecao sob SCAR dentro do PU classico;
- **Trilha C**: formulacao mais moderna, risk-based, para repensar o problema alem de SCAR.

## 1) Setup do ambiente

Nesta secao, serao definidos imports, versoes, configuracao de SMOKE, semente global e utilitarios para diagnostico, ranking, interpretabilidade, sensibilidade estrutural e mascaramento de CPF nas saidas exibidas.

In [ ]:
# ---------------------------------------------------------
# Setup tecnico do experimento PU (reprodutibilidade)
# ---------------------------------------------------------
# Contexto:
# Esta celula centraliza imports, configuracoes globais e funcoes utilitarias
# usadas em todo o notebook.
#
# Objetivo estatistico/operacional:
# Garantir reproducibilidade (semente fixa), rastreabilidade de versoes e
# validacao defensiva dos caminhos de entrada/saida.
#
# Interpretacao:
# O bloco de print final funciona como log de auditoria do ambiente.
# Em contexto de artigo, ele permite replicar o experimento no mesmo stack.

import io
import os
import sys
import time
import random
import platform
import warnings
import math
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import pulearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


SMOKE = True
SMOKE_MAX_ROWS = 300_000

TOP_N_RANKING = 100_000
TOP_RANK_PCTS = [0.01, 0.05, 0.10]
TOP_RANK_ABS = [1000, 5000]
PLOT_MAX_DISPLAY = 20
MIN_GROUP_ROWS = 500
MAX_SHAP_ROWS_GLOBAL = 20_000 if SMOKE else 120_000
MAX_SHAP_ROWS_TOP = 20_000 if SMOKE else 80_000
HOLD_OUT_RATIO_MAIN = 0.10
HOLD_OUT_RATIO_GRID = [0.05, 0.10, 0.15, 0.20]

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "dados" / "dados_processados" / "df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet"
OUTPUT_DIR = ROOT / "dados" / "dados_processados" / "outputs_pu"

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    xgb_import_error = None
except Exception as exc:
    xgb = None
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

    xgb_import_error = str(exc)

try:
    import shap
    SHAP_AVAILABLE = True
    shap_import_error = None
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    shap_import_error = str(exc)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo de entrada nao encontrado: {DATA_PATH}")


def memory_mb(df_ref: pd.DataFrame) -> float:
    return float(df_ref.memory_usage(deep=True).sum() / 1024**2)


@contextmanager
def timer(label: str):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    print(f"[tempo] {label}: {dt:.2f}s")


def resumo_nulos(df_ref: pd.DataFrame, cols: list[str] | None = None, top_n: int = 15) -> pd.DataFrame:
    if cols is None:
        cols = list(df_ref.columns)
    total = len(df_ref)
    out = (
        df_ref[cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
        .to_frame("n_missing")
    )
    out["pct_missing"] = out["n_missing"] / total
    return out.head(top_n)


def get_positive_class_index(pu_step, positive_label: int = 1) -> tuple[int, str, list]:
    candidates = []

    if hasattr(pu_step, "classes_"):
        candidates.append(("pu_step.classes_", np.asarray(pu_step.classes_)))

    if hasattr(pu_step, "estimator_") and hasattr(pu_step.estimator_, "classes_"):
        candidates.append(("pu_step.estimator_.classes_", np.asarray(pu_step.estimator_.classes_)))

    for source_name, classes in candidates:
        idx = np.where(classes == positive_label)[0]
        if len(idx) > 0:
            return int(idx[0]), source_name, classes.tolist()

    raise ValueError(
        "Nao foi possivel mapear a classe positiva (1) em classes_. "
        "Sem esse mapeamento, o score de predict_proba nao e confiavel."
    )


def sample_smoke_by_group(df_ref: pd.DataFrame, group_col: str, max_rows: int, seed: int) -> tuple[pd.DataFrame, dict]:
    if group_col not in df_ref.columns:
        raise ValueError(f"Coluna de grupo ausente para SMOKE: {group_col}")

    n_rows_original = int(len(df_ref))
    n_groups_original = int(df_ref[group_col].nunique())

    if n_rows_original <= max_rows:
        return df_ref.reset_index(drop=True).copy(), {
            "applied": False,
            "n_rows_original": n_rows_original,
            "n_rows_final": n_rows_original,
            "n_groups_original": n_groups_original,
            "n_groups_final": n_groups_original,
            "criterion": "dataset_already_below_limit",
            "seed": int(seed),
        }

    group_sizes = df_ref.groupby(group_col, sort=False).size()
    shuffled_groups = group_sizes.index.to_numpy(copy=True)
    rng = np.random.default_rng(seed)
    rng.shuffle(shuffled_groups)

    selected_groups = []
    cumulative_rows = 0
    for group_value in shuffled_groups:
        group_n = int(group_sizes.loc[group_value])
        if cumulative_rows + group_n > max_rows and selected_groups:
            continue
        if cumulative_rows + group_n > max_rows and not selected_groups:
            selected_groups.append(group_value)
            cumulative_rows += group_n
            break
        selected_groups.append(group_value)
        cumulative_rows += group_n
        if cumulative_rows >= max_rows:
            break

    df_out = df_ref[df_ref[group_col].isin(selected_groups)].copy().reset_index(drop=True)

    return df_out, {
        "applied": True,
        "n_rows_original": n_rows_original,
        "n_rows_final": int(len(df_out)),
        "n_groups_original": n_groups_original,
        "n_groups_final": int(df_out[group_col].nunique()),
        "criterion": "group_preserving_random_sample",
        "seed": int(seed),
    }


def predict_positive_score(model_pipeline, X_ref, positive_label: int = 1) -> tuple[np.ndarray, int, str, list]:
    if "pu" not in model_pipeline.named_steps:
        raise ValueError("Pipeline sem etapa 'pu'; nao e possivel extrair score positivo.")

    pu_step = model_pipeline.named_steps["pu"]
    pos_idx, class_source, class_labels = get_positive_class_index(pu_step, positive_label=positive_label)

    proba = np.asarray(model_pipeline.predict_proba(X_ref))
    if proba.ndim != 2 or proba.shape[1] <= pos_idx:
        raise ValueError(f"Saida inesperada de predict_proba: shape={proba.shape}, pos_idx={pos_idx}")

    score_raw = np.nan_to_num(
        proba[:, pos_idx].astype("float64"),
        nan=0.0,
        posinf=np.finfo("float64").max,
        neginf=0.0,
    )

    return score_raw, pos_idx, class_source, class_labels


def compute_ranking_metrics(y_true, scores, k_fracoes: tuple[float, ...] = (0.01, 0.05, 0.10)) -> tuple[float, pd.DataFrame]:
    y_np = pd.Series(y_true).astype("int8").to_numpy(copy=False)
    score_np = pd.to_numeric(pd.Series(scores), errors="coerce").fillna(0.0).to_numpy(dtype="float64", copy=False)

    if len(y_np) != len(score_np):
        raise ValueError(f"Tamanho inconsistente entre y_true ({len(y_np)}) e scores ({len(score_np)}).")

    positivos_totais = int((y_np == 1).sum())
    if positivos_totais == 0:
        raise ValueError("Nao ha positivos em y_true para calcular metricas de ranking.")

    auc_proxy = roc_auc_score(y_np, score_np)
    ord_desc = np.argsort(-score_np)

    rows = []
    for frac in k_fracoes:
        k = max(int(np.ceil(frac * len(score_np))), 1)
        topk_idx = ord_desc[:k]
        positivos_topk = int((y_np[topk_idx] == 1).sum())
        rows.append({
            "k_frac": float(frac),
            "k_absoluto": int(k),
            "positivos_topk": int(positivos_topk),
            "positivos_totais_test": int(positivos_totais),
            "recall_at_k": float(positivos_topk / positivos_totais),
            "precision_at_k": float(positivos_topk / k),
        })

    return float(auc_proxy), pd.DataFrame(rows)


def metric_at(df_metricas: pd.DataFrame, frac: float, metric_col: str) -> float:
    return float(df_metricas.loc[df_metricas["k_frac"] == frac, metric_col].iloc[0])


def topk_overlap(df_ranked: pd.DataFrame, score_col_a: str, score_col_b: str, frac: float) -> float:
    if len(df_ranked) == 0:
        return float("nan")

    k = max(int(np.ceil(frac * len(df_ranked))), 1)
    ids_a = set(df_ranked.sort_values([score_col_a, "cpf_agricultor"], ascending=[False, True]).head(k)["cpf_agricultor"].astype(str).tolist())
    ids_b = set(df_ranked.sort_values([score_col_b, "cpf_agricultor"], ascending=[False, True]).head(k)["cpf_agricultor"].astype(str).tolist())
    return float(len(ids_a & ids_b) / k)



def mask_cpf_value(value) -> str | pd._libs.missing.NAType:
    if pd.isna(value):
        return pd.NA

    digits = ''.join(ch for ch in str(value) if ch.isdigit())
    if len(digits) >= 11:
        digits = digits[-11:]

    if len(digits) < 6:
        return str(value)

    return f"{digits[:3]}*****{digits[-3:]}"


def mask_cpf_series(series: pd.Series) -> pd.Series:
    return series.astype('string').map(mask_cpf_value)


def mask_cpf_frame(df_ref: pd.DataFrame, columns: list[str] | tuple[str, ...] | None = None) -> pd.DataFrame:
    if columns is None:
        columns = ['cpf_agricultor']

    df_out = df_ref.copy()
    for col in columns:
        if col in df_out.columns:
            df_out[col] = mask_cpf_series(df_out[col])
    return df_out


def topn_overlap(df_ranked: pd.DataFrame, score_col_a: str, score_col_b: str, top_n: int) -> float:
    if len(df_ranked) == 0:
        return float('nan')

    k = max(int(min(top_n, len(df_ranked))), 1)
    ids_a = set(
        df_ranked.sort_values([score_col_a, 'cpf_agricultor'], ascending=[False, True])
        .head(k)['cpf_agricultor']
        .astype(str)
        .tolist()
    )
    ids_b = set(
        df_ranked.sort_values([score_col_b, 'cpf_agricultor'], ascending=[False, True])
        .head(k)['cpf_agricultor']
        .astype(str)
        .tolist()
    )
    return float(len(ids_a & ids_b) / k)


def safe_corr(series_a, series_b, method: str = 'pearson') -> float:
    df_corr = pd.DataFrame({'a': series_a, 'b': series_b}).dropna()
    if len(df_corr) < 2:
        return float('nan')
    return float(df_corr['a'].corr(df_corr['b'], method=method))


def compute_segment_shap_xgb(
    df_segment: pd.DataFrame,
    segment_name: str,
    booster,
    feature_names: list[str],
    max_rows: int | None = None,
    seed: int = 42,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, pd.DataFrame]:
    if not XGBOOST_AVAILABLE or xgb is None:
        raise ImportError('xgboost nao esta disponivel para o bloco SHAP.')

    if booster is None:
        raise ValueError('Booster XGBoost ausente para o bloco SHAP.')

    if len(df_segment) == 0:
        raise ValueError(f'Segmento SHAP vazio: {segment_name}')

    feature_names = [str(col) for col in feature_names]

    if max_rows is not None and len(df_segment) > int(max_rows):
        df_used = df_segment.sample(n=int(max_rows), random_state=seed).copy().reset_index(drop=True)
        amostrado = True
    else:
        df_used = df_segment.copy().reset_index(drop=True)
        amostrado = False

    t0_segment = time.perf_counter()
    dmatrix = xgb.DMatrix(
        df_used[feature_names].astype('float32'),
        feature_names=feature_names,
    )
    pred_contribs = booster.predict(
        dmatrix,
        pred_contribs=True,
        validate_features=False,
    )
    elapsed_segment = time.perf_counter() - t0_segment

    shap_values = np.asarray(pred_contribs[:, :-1], dtype='float64')
    base_values = np.asarray(pred_contribs[:, -1], dtype='float64')

    df_importance = pd.DataFrame(
        {
            'feature': feature_names,
            'mean_abs_shap': np.abs(shap_values).mean(axis=0),
            'mean_shap': shap_values.mean(axis=0),
            'segmento': segment_name,
            'n_linhas_segmento': int(len(df_segment)),
            'n_linhas_usadas_shap': int(len(df_used)),
            'amostrado': bool(amostrado),
            'tempo_segundos': float(elapsed_segment),
        }
    ).sort_values('mean_abs_shap', ascending=False, ignore_index=True)

    return df_used, shap_values, base_values, df_importance

print("Ambiente:")
print(f"- Python: {sys.version.split()[0]}")
print(f"- Plataforma: {platform.platform()}")
print(f"- pandas: {pd.__version__}")
print(f"- numpy: {np.__version__}")
print(f"- scikit-learn: {sklearn.__version__}")
print(f"- pulearn: {pulearn.__version__}")
print(f"- SEED global: {SEED}")
print(f"- DATA_PATH: {DATA_PATH}")
print(f"- OUTPUT_DIR: {OUTPUT_DIR}")
print(f"- SMOKE: {SMOKE}")
print(f"- SMOKE_MAX_ROWS: {SMOKE_MAX_ROWS}")
print(f"- XGBoost disponivel para B2: {XGBOOST_AVAILABLE}")
print(f"- SHAP disponivel para plots: {SHAP_AVAILABLE}")
print(f"- hold_out_ratio principal: {HOLD_OUT_RATIO_MAIN:.2f}")
print(f"- grade hold_out_ratio: {HOLD_OUT_RATIO_GRID}")
if not XGBOOST_AVAILABLE and xgb_import_error is not None:
    print(f"- Motivo do fallback potencial para HGB: {xgb_import_error}")


## 2) Carregamento dos dados

Nesta secao, sera feita a leitura do parquet base, uma inspecao inicial de shape, tipos e distribuicao de `s` e, quando `SMOKE=True`, uma reducao reprodutivel do volume para acelerar testes da trilha.


In [ ]:
# ---------------------------------------------------------
# Carregamento e diagnostico inicial do dataset consolidado
# ---------------------------------------------------------
# Contexto:
# Leitura do parquet principal da Trilha B em nivel pessoa.
#
# Objetivo estatistico:
# Verificar dimensao, tipagem e distribuicao do rotulo observado s.
# Isso confirma se o problema PU esta representado como esperado.
#
# Interpretacao:
# A distribuicao de s (1 vs 0) nao mede prevalencia real de y, mas orienta
# expectativa de dificuldade do ranking sob rotulo parcial.

with timer("Leitura do parquet principal"):
    df = pd.read_parquet(DATA_PATH, engine="pyarrow")

print(f"Shape: {df.shape}")
print(f"Colunas totais: {len(df.columns)}")
print(f"Memoria aproximada do dataframe: {memory_mb(df):.1f} MB")

if "s" not in df.columns:
    raise ValueError("Coluna obrigatoria ausente: 's'")

print("\nDtypes por coluna:")
display(df.dtypes.to_frame(name="dtype"))

key_cols = [c for c in ["cpf_agricultor", "id_familia", "ibge", "s"] if c in df.columns]

if "id_familia" not in df.columns:
    raise ValueError("Coluna obrigatoria ausente para SMOKE por grupo: 'id_familia'")

if SMOKE:
    df, smoke_info = sample_smoke_by_group(df_ref=df, group_col="id_familia", max_rows=SMOKE_MAX_ROWS, seed=SEED)
else:
    smoke_info = {
        "applied": False,
        "n_rows_original": int(len(df)),
        "n_rows_final": int(len(df)),
        "n_groups_original": int(df["id_familia"].nunique()),
        "n_groups_final": int(df["id_familia"].nunique()),
        "criterion": "smoke_disabled",
        "seed": int(SEED),
    }

print("\nConfiguracao SMOKE:")
print(f"- ativo: {SMOKE}")
print(f"- limite de linhas: {SMOKE_MAX_ROWS:,}")
print(f"- seed usada: {smoke_info['seed']}")
print(f"- criterio: {smoke_info['criterion']}")
print(f"- linhas originais: {smoke_info['n_rows_original']:,}")
print(f"- linhas apos smoke: {smoke_info['n_rows_final']:,}")
print(f"- familias originais: {smoke_info['n_groups_original']:,}")
print(f"- familias apos smoke: {smoke_info['n_groups_final']:,}")
if smoke_info["applied"]:
    print("- Observacao: SMOKE serve apenas para teste rapido do pipeline e nao substitui resultados finais full.")

print(f"\nShape apos eventual SMOKE: {df.shape}")
print(f"Memoria aproximada apos eventual SMOKE: {memory_mb(df):.1f} MB")

print("\nMissing em colunas-chave:")
display(df[key_cols].isna().sum().to_frame(name="n_missing"))

cardinalidade_rows = []
for col in ["cpf_agricultor", "id_familia", "ibge"]:
    if col in df.columns:
        cardinalidade_rows.append(
            {
                "coluna": col,
                "n_unicos": int(df[col].nunique(dropna=True)),
                "n_missing": int(df[col].isna().sum()),
            }
        )

print("\nCardinalidade das chaves analiticas:")
display(pd.DataFrame(cardinalidade_rows))

s_counts = df["s"].value_counts(dropna=False).sort_index()
s_percent = (s_counts / len(df)).rename("pct")
resumo_s = pd.concat([s_counts.rename("count"), s_percent], axis=1)

print("\nDistribuicao de s (count e %):")
display(resumo_s)

main_cols = [c for c in ["cpf_agricultor", "id_familia", "ibge", "s"] if c in df.columns]
print("\nHead com colunas principais:")
display(mask_cpf_frame(df[main_cols].head(), columns=["cpf_agricultor"]))

info_buffer = io.StringIO()
df.info(buf=info_buffer)
print("\nResumo estrutural via df.info():")
print(info_buffer.getvalue())


### Interpretacao da Secao 2 (carregamento)

**Contexto do bloco**
- O arquivo carregado e o consolidado da Trilha B (`df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet`).
- A granularidade e **nivel pessoa** (um agricultor por linha), com variaveis de familia replicadas por `id_familia`.

**Objetivo estatistico**
- Confirmar que o alvo PU (`s`) esta disponivel e com distribuicao plausivel para modelagem.
- Registrar qualidade minima da base antes de qualquer exclusao de features ou split.

**Como interpretar os resultados mostrados**
- `df.shape` e memoria indicam custo computacional do baseline.
- `df.dtypes` e `df.info()` funcionam como fotografia do schema efetivamente usado.
- Missing em `cpf_agricultor`, `id_familia`, `ibge` e `s` sinalizam risco direto de quebra de granularidade ou de rastreabilidade.
- A cardinalidade das chaves ajuda a verificar se a base continua coerente com a expectativa de nivel pessoa.
- `value_counts` de `s` mostra o desbalanceamento natural de PU (poucos `s=1`, muitos `s=0`).


## 3) Definição de features e alvo PU

Nesta seção, são definidos `TARGET_COL`, `GROUP_COL`, `ID_COLS` e a lista de features candidatas.

`s=0` representa **unlabeled**: esse grupo mistura exemplos realmente positivos (`y=1`) e negativos (`y=0`).
Por isso, o objetivo do baseline é produzir **ranking de propensão**, e não tratar `s=0` como negativo puro.


In [ ]:
# ---------------------------------------------------------
# Definicao formal de alvo PU, grupos e conjunto de features
# ---------------------------------------------------------
# Contexto:
# Delimita as colunas obrigatorias para modelagem e valida integridade minima.
#
# Objetivo estatistico:
# Assegurar que s esteja em {0,1}, excluir IDs do modelo e preservar ibge como
# metadado analitico (fora do baseline preditivo da Trilha B).
#
# Interpretacao:
# Essa etapa reduz risco de vazamento e explicita que s=0 e unlabeled
# (mistura de y=0 e y=1), portanto o resultado principal sera ranking.

TARGET_COL = "s"
GROUP_COL = "id_familia"
ID_COLS = ["cpf_agricultor", "id_familia"]

# Trilha B: ibge fora do modelo; mantido apenas para analise/output
META_COLS = ["ibge"]
EXCLUDED_FROM_MODEL = [TARGET_COL] + ID_COLS + META_COLS

required_cols = set([TARGET_COL, GROUP_COL] + ID_COLS + META_COLS)
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Colunas obrigatorias ausentes: {missing_cols}")

s_numeric = pd.to_numeric(df[TARGET_COL], errors="coerce")
if s_numeric.isna().any():
    n_nan = int(s_numeric.isna().sum())
    raise ValueError(f"Foram encontrados {n_nan} valores nao numericos em s.")

invalid_mask = ~s_numeric.isin([0, 1])
if invalid_mask.any():
    invalid_values = sorted(s_numeric[invalid_mask].unique().tolist())
    raise ValueError(f"s contem valores fora de {{0,1}}: {invalid_values[:10]}")

df[TARGET_COL] = s_numeric.astype("int8")

feature_cols = [c for c in df.columns if c not in EXCLUDED_FROM_MODEL]
if not feature_cols:
    raise ValueError("Nenhuma feature disponivel apos remover alvo, IDs e META_COLS.")

print("Configuracao de modelagem:")
print(f"- TARGET_COL: {TARGET_COL}")
print(f"- GROUP_COL: {GROUP_COL}")
print(f"- ID_COLS: {ID_COLS}")
print(f"- META_COLS (fora do modelo): {META_COLS}")
print(f"- Colunas explicitamente excluidas do modelo: {EXCLUDED_FROM_MODEL}")
print(f"- Total de colunas no dataframe: {len(df.columns)}")
print(f"- Features candidatas no modelo: {len(feature_cols)}")

print("\nInterpretacao PU:")
print("- s=1: positivo rotulado")
print("- s=0: unlabeled (mistura de positivos reais e negativos)")
print("- IDs e metadados territoriais ficam fora do modelo para reduzir risco de vazamento")

numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
non_numeric_feature_cols = [c for c in feature_cols if c not in numeric_feature_cols]

print("\nTipos de feature para modelagem:")
print(f"- Numericas: {len(numeric_feature_cols)}")
print(f"- Nao numericas (fora da Trilha B): {len(non_numeric_feature_cols)}")
if non_numeric_feature_cols:
    print(f"- Exemplo de nao numericas: {non_numeric_feature_cols[:10]}")


### Interpretacao da Secao 3 (features e alvo)

**Decisao metodologica central**
- `cpf_agricultor` e `id_familia` sao identificadores e nao entram como preditores para evitar vazamento direto.
- `ibge` fica fora do modelo na Trilha B por decisao de baseline. Ele permanece apenas como metadado para leitura territorial posterior, sem influenciar o score nesta trilha.
- Features nao numericas tambem ficam fora do baseline atual para manter o experimento simples, reprodutivel e comparavel.

**Objetivo estatistico**
- Garantir que `s` esteja estritamente em `{0,1}` antes do treino.
- Definir uma matriz de features coerente com o escopo do baseline rapido.

**Leitura analitica**
- `s=0` e unlabeled (mistura de `y=0` e `y=1`), entao o modelo produz **ranking de propensao**, nao probabilidade causal definitiva.
- A lista de colunas excluidas ajuda a auditar o que foi retirado por risco de vazamento ou por escolha deliberada de baseline.


## 4) Tratamento de dados (nulos e tipos)

Neste baseline, usamos pré-processamento simples e reprodutível: **imputação por mediana para variáveis numéricas**.

Não aplicamos SMOTE, `class_weight` ou outras técnicas de balanceamento nesta trilha inicial.
Motivo: no Elkan–Noto, a correção PU já ocorre pelo mecanismo do método (estimativa de `c` no holdout).
Adicionar balanceamento agora dificultaria interpretar o comportamento do baseline.


In [ ]:
# ---------------------------------------------------------
# Pre-processamento numerico minimo e robusto para baseline
# ---------------------------------------------------------
# Contexto:
# Implementa pipeline simples para dados numericos com nulos.
#
# Objetivo estatistico:
# Usar imputacao por mediana e padronizacao para estabilidade numerica da
# regressao logistica no wrapper Elkan-Noto.
#
# Interpretacao:
# Nao ha balanceamento artificial nesta trilha; o baseline busca leitura limpa
# do metodo PU antes de tecnicas adicionais de engenharia.

# Opcional e leve: flags de missing em lat/lon
lat_lon_flags = []
for col in ["lat_imovel_principal", "lon_imovel_principal"]:
    if col in df.columns:
        flag_col = f"st_missing_{col}"
        if flag_col not in df.columns:
            df[flag_col] = df[col].isna().astype("int8")
        lat_lon_flags.append(flag_col)

# Regra consolidada (unica) de features da Trilha B
feature_cols = [c for c in df.columns if c not in ([TARGET_COL] + ID_COLS + META_COLS)]
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
non_numeric_feature_cols = [c for c in feature_cols if c not in numeric_feature_cols]

if len(numeric_feature_cols) == 0:
    raise ValueError("Nenhuma feature numerica disponivel para o baseline.")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_feature_cols,
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Pre-processamento definido:")
print("- Estrategia numerica: imputacao por mediana + StandardScaler")
print(f"- Features numericas usadas: {len(numeric_feature_cols)}")
print(f"- Features nao numericas fora do baseline: {len(non_numeric_feature_cols)}")
print(f"- Regra META_COLS fora do modelo: {META_COLS}")
if lat_lon_flags:
    print(f"- Flags de missing criadas para lat/lon: {lat_lon_flags}")

print("\nResumo de nulos (top 15):")
display(resumo_nulos(df, cols=numeric_feature_cols, top_n=15))


### Interpretacao da Secao 4 (tratamento de nulos e tipos)

**Por que esta transformacao e necessaria**
- Imputacao por mediana evita perda de linhas e mantem robustez para caudas/outliers.
- Padronizacao (`StandardScaler`) melhora estabilidade numerica e convergencia da regressao logistica (`solver='saga'`).

**Decisao de baseline**
- Nao usar SMOTE, `class_weight` ou rebalanceamento nesta trilha.
- Motivo: o metodo Elkan-Noto ja incorpora ajuste PU via estimativa interna de `c`; adicionar balanceamento nesta etapa confundiria a interpretacao do baseline.

**Como interpretar o resumo de nulos**
- O top de missing identifica variaveis com maior risco de ruido e orienta melhorias para trilhas seguintes.


### Extra - avaliação por família

In [ ]:
# ---------------------------------------------------------
# Diagnostico complementar da estrutura por familia/pessoa
# ---------------------------------------------------------
# Objetivo:
# Entender a granularidade da base e a distribuicao do rotulo observado s
# nas unidades de agrupamento relevantes para o split.
#
# Interpretacao:
# - A media da proporcao de s=1 por familia nao e a prevalencia global de s=1;
#   ela da peso igual para cada familia, independentemente do tamanho.
# - A proporcao de familias com ao menos um s=1 ajuda a entender a cobertura
#   observada do rotulo positivo no nivel familiar.

In [ ]:
# número de id_familia únicos
n_familias = df["id_familia"].nunique()
print(f"Número de id_familia únicos: {n_familias}")

# número de cpf_agricultor únicos
n_agricultores = df["cpf_agricultor"].nunique()
print(f"Número de cpf_agricultor únicos: {n_agricultores}")

In [ ]:
# taxa média de positivos por família
#média não ponderada da proporção de positivos rotulados dentro das famílias.
taxa_positivos_por_familia = df.groupby("id_familia")[TARGET_COL].mean()
print(f"Taxa média de positivos por família: {taxa_positivos_por_familia.mean():.4f}")      

# taxa média de positivos por cpf_agricultor
taxa_positivos_por_agricultor = df.groupby("cpf_agricultor")[TARGET_COL].mean()
print(f"Taxa média de positivos por cpf_agricultor: {taxa_positivos_por_agricultor.mean():.4f}")

In [ ]:
# proporção de famílias que contêm ao menos um s=1
familias_com_positivo = df.groupby("id_familia")[TARGET_COL].max()
proporcao_familias_com_positivo = (familias_com_positivo == 1).mean()
print(f"Proporção de famílias com pelo menos um s=1: {proporcao_familias_com_positivo:.4f}")    



## 5) Split treino/validacao/teste (com grupos por familia)

Para evitar vazamento entre membros da mesma familia, o particionamento e feito por `id_familia` com `GroupShuffleSplit`.

Estrategia reproduzivel em duas etapas:
1. `train` vs `temp` com 80% / 20%.
2. `temp` em `val` e `test` com 50% / 50% (resultando em 10% / 10% do total).

Observacao metodologica importante:
- este split e **sensivel a grupos**, mas nao e estratificado simultaneamente por `s`;
- por isso, pequenas diferencas na prevalencia observada de `s=1` entre particoes sao aceitaveis, desde que permanecam proximas da distribuicao global.


In [ ]:
# ---------------------------------------------------------
# Split por grupos (id_familia) para evitar vazamento
# ---------------------------------------------------------
# Contexto:
# Particionamento em duas etapas: train (~80%), val (~10%), test (~10%),
# respeitando a unidade familiar.
#
# Objetivo estatistico:
# Impedir que membros da mesma familia aparecam em particoes diferentes,
# preservando validade da avaliacao fora de amostra.
#
# Interpretacao:
# Overlap entre grupos deve ser zero; pequenas diferencas nas proporcoes de s=1
# entre splits podem ocorrer, pois o procedimento nao e estratificado por alvo.

X_all = df[feature_cols]
y_all = df[TARGET_COL].astype("int8")
groups_all = df[GROUP_COL].astype("string")
pct_s1_global = float(y_all.mean())

if groups_all.isna().any():
    n_missing_group = int(groups_all.isna().sum())
    raise ValueError(f"{GROUP_COL} possui {n_missing_group} valores ausentes; split por grupos nao e seguro.")

# Etapa 1: train (~80%) / temp (~20%)
gss_stage1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    test_size=0.2,
    random_state=SEED,
)
idx_train, idx_temp = next(gss_stage1.split(X_all, y=y_all, groups=groups_all))

# Etapa 2: temp -> val (~10%) / test (~10%)
X_temp = X_all.iloc[idx_temp]
y_temp = y_all.iloc[idx_temp]
groups_temp = groups_all.iloc[idx_temp]

gss_stage2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.5,
    test_size=0.5,
    random_state=SEED,
)
idx_val_in_temp, idx_test_in_temp = next(
    gss_stage2.split(X_temp, y=y_temp, groups=groups_temp)
)

idx_val = idx_temp[idx_val_in_temp]
idx_test = idx_temp[idx_test_in_temp]

# Checagem de vazamento por grupo
train_groups = pd.unique(groups_all.iloc[idx_train])
val_groups = pd.unique(groups_all.iloc[idx_val])
test_groups = pd.unique(groups_all.iloc[idx_test])

overlap_train_val = len(np.intersect1d(train_groups, val_groups))
overlap_train_test = len(np.intersect1d(train_groups, test_groups))
overlap_val_test = len(np.intersect1d(val_groups, test_groups))

assert overlap_train_val == 0, "Vazamento detectado: familias em train e val"
assert overlap_train_test == 0, "Vazamento detectado: familias em train e test"
assert overlap_val_test == 0, "Vazamento detectado: familias em val e test"

# Checagem adicional de sobreposicao de linhas
assert len(np.intersect1d(idx_train, idx_val)) == 0, "Sobreposicao de linhas entre train e val"
assert len(np.intersect1d(idx_train, idx_test)) == 0, "Sobreposicao de linhas entre train e test"
assert len(np.intersect1d(idx_val, idx_test)) == 0, "Sobreposicao de linhas entre val e test"


def resumo_split(nome: str, idx: np.ndarray) -> dict:
    y_split = y_all.iloc[idx]
    grupos_split = groups_all.iloc[idx]
    counts = y_split.value_counts().reindex([0, 1], fill_value=0)
    n_rows = len(idx)
    n_groups = int(grupos_split.nunique())
    pct_s_1 = float(counts.loc[1] / n_rows) if n_rows > 0 else 0.0

    return {
        "split": nome,
        "n_linhas": n_rows,
        "pct_linhas": float(n_rows / len(df)),
        "n_familias": n_groups,
        "pct_familias": float(n_groups / groups_all.nunique()),
        "s_0": int(counts.loc[0]),
        "s_1": int(counts.loc[1]),
        "pct_s_1": pct_s_1,
        "delta_pct_s_1_vs_global": float(pct_s_1 - pct_s1_global),
    }


resumo_splits = pd.DataFrame(
    [
        resumo_split("train", idx_train),
        resumo_split("val", idx_val),
        resumo_split("test", idx_test),
    ]
)

print("Split por grupos concluido com sucesso.")
print(f"Total de linhas: {len(df):,}")
print(f"Linhas train/val/test: {len(idx_train):,} / {len(idx_val):,} / {len(idx_test):,}")
print(f"Prevalencia global de s=1: {pct_s1_global:.6f}")
print(
    "Overlaps de familia (devem ser 0): "
    f"train-val={overlap_train_val}, train-test={overlap_train_test}, val-test={overlap_val_test}"
)

print("\nResumo dos splits:")
display(resumo_splits)


### Interpretacao da Secao 5 (split por grupos)

**Objetivo estatistico**
- Evitar vazamento entre treino/validacao/teste quando membros de uma mesma familia compartilham atributos.

**Operacao computacional**
- Split em duas etapas com `GroupShuffleSplit` usando `id_familia`:
  1. 80% treino / 20% temporario.
  2. 10% validacao / 10% teste a partir do temporario.

**Interpretacao esperada**
- Overlaps entre grupos devem ser zero.
- A distribuicao de `s` entre particoes deve ser proxima da distribuicao global para reduzir vies de avaliacao.
- Como o split nao e estratificado por `s`, o campo `delta_pct_s_1_vs_global` serve para auditar a distancia entre cada particao e a prevalencia observada no dataset completo.


## 6) B1 - Elkan-Noto + Logistic Regression

Nesta secao, congelamos o baseline metodologico principal da Trilha B, executamos tuning controlado de convergencia e selecionamos a melhor configuracao segundo um **criterio conservador de elegibilidade numerica**.

O parametro `hold_out_ratio` permanece fixado em `HOLD_OUT_RATIO_MAIN (= 0.10)` na execucao principal para reservar parte dos positivos rotulados do treino e estimar `c = P(s=1|y=1)`.
A escolha final primeiro restringe os candidatos a configuracoes com `converged=True` e, entre elas, usa AUC proxy em validacao e custo de treino para desempate.

Leitura metodologica importante:
- o classificador base aprende inicialmente um sinal ligado a `s`;
- a constante `c` corrige essa saida sob a hipotese SCAR para produzir score associado ao alvo latente `y`;
- a sensibilidade de `hold_out_ratio` deixa de ser apenas observacao lateral e passa a ser tratada explicitamente em secao propria de robustez.

In [ ]:
# ---------------------------------------------------------
# Treino comparativo: baseline congelado + tuning controlado
# ---------------------------------------------------------
# Contexto:
# Esta celula executa a etapa central de comparacao entre a configuracao
# baseline da Trilha B e um pequeno conjunto de configuracoes alternativas
# da regressao logistica usada como estimador base do modelo PU.
#
# Objetivo estatistico:
# Selecionar uma configuracao com boa separacao entre s=1 e s=0 na validacao,
# sob restricoes de estabilidade numerica e custo computacional.
#
# Justificativa metodologica:
# No baseline de Elkan & Noto, o classificador base aprende uma aproximacao
# para P(s=1 | x), e essa saida e depois ajustada pela constante c para gerar
# scores associados a P(y=1 | x). Assim, nesta etapa, o criterio de selecao
# da configuracao pode ser baseado em desempenho sobre o rotulo observado s.
#
# Criterio proxy:
# AUC em s na validacao e usada como metrica proxy de capacidade discriminativa.
# Ela nao mede diretamente a qualidade em y (latente), mas informa se o modelo
# consegue ordenar melhor positivos rotulados versus unlabeled.
#
# Estrategia operacional:
# O tuning e feito em uma amostra controlada de train/val para reduzir custo.
# Como o objetivo aqui nao e hiperotimizacao exaustiva, mas apenas checar
# robustez do baseline, esse desenho preserva rastreabilidade sem tornar o
# notebook excessivamente caro.
#
# Interpretacao:
# O dataframe df_model_compare registra a trilha de decisao metodologica:
# desempenho proxy, convergencia, numero de iteracoes, tempo de treino,
# estimativa de c e configuracao escolhida para a etapa final.

# ---------------------------------------------------------
# Particoes base ja definidas no split por grupos
# ---------------------------------------------------------
# Objetivo:
# Materializar os subconjuntos de treino, validacao e teste a partir dos
# indices previamente obtidos com GroupShuffleSplit.
#
# Interpretacao:
# Essas particoes respeitam a restricao de nao compartilhar familias entre si,
# reduzindo risco de vazamento entre observacoes correlacionadas.

X_train = X_all.iloc[idx_train]
y_train = y_all.iloc[idx_train]
X_val = X_all.iloc[idx_val]
y_val = y_all.iloc[idx_val]
X_test = X_all.iloc[idx_test]
y_test = y_all.iloc[idx_test]

if y_train.nunique() < 2:
    raise ValueError("Treino sem as duas classes de s (0 e 1).")

# ---------------------------------------------------------
# Baseline congelado para comparacao
# ---------------------------------------------------------
# Objetivo:
# Registrar explicitamente a configuracao original da Trilha B, de modo que
# qualquer ganho posterior possa ser comparado contra um ponto de referencia
# auditavel.
#
# Interpretacao:
# Esse baseline nao e apenas um chute inicial; ele funciona como marco
# metodologico da trilha e deve permanecer visivel no quadro comparativo.

baseline_config = {
    "model_name": "baseline_original", # identificacao clara para o baseline congelado
    "C": 1.0, # valor padrao de regularizacao l2 na LogisticRegression, usado como referencia
    "max_iter": 1000, # valor padrao de max_iter na LogisticRegression, escolhido para tentar garantir convergencia sem ser excessivo
}

# ---------------------------------------------------------
# Tuning controlado em amostra
# ---------------------------------------------------------
# Objetivo:
# Reduzir custo computacional da fase comparativa sem comprometer a leitura
# relativa entre configuracoes candidatas.
#
# Justificativa:
# Em bases muito grandes, a comparacao entre poucas configuracoes tende a
# preservar a ordenacao relativa mesmo quando feita em subamostra, desde que
# a amostra seja suficientemente grande e sorteada de forma reprodutivel.
#
# Interpretacao:
# Esta etapa nao substitui o treino final; ela apenas seleciona configuracoes
# mais promissoras para o baseline, evitando custo excessivo.

rng = np.random.default_rng(SEED)

tune_train_size = min(120_000, len(idx_train)) # tamanho controlado para tuning, ajustado para caber na memoria e ser rapido, mas ainda representativo
tune_val_size = min(120_000, len(idx_val))

if tune_train_size < len(idx_train):
    idx_train_tune = np.sort(rng.choice(idx_train, size=tune_train_size, replace=False))
else:
    idx_train_tune = idx_train

if tune_val_size < len(idx_val):
    idx_val_tune = np.sort(rng.choice(idx_val, size=tune_val_size, replace=False))
else:
    idx_val_tune = idx_val

X_train_tune = X_all.iloc[idx_train_tune]
y_train_tune = y_all.iloc[idx_train_tune]
X_val_tune = X_all.iloc[idx_val_tune]
y_val_tune = y_all.iloc[idx_val_tune].to_numpy(dtype="int8", copy=False)

# ---------------------------------------------------------
# Conjunto reduzido de configuracoes candidatas
# ---------------------------------------------------------
# Objetivo:
# Testar variacoes controladas de regularizacao (C) e orcamento de iteracoes
# (max_iter), sem transformar a Trilha B em uma busca extensa de hiperparametros.
#
# Interpretacao:
# O foco aqui e robustez do baseline. Variar poucos pontos bem escolhidos ajuda
# a identificar se o modelo esta sensivel a convergencia ou regularizacao sem
# descaracterizar a proposta de baseline simples.

candidate_configs = [
    baseline_config,

    {"model_name": "cfg_c0_1_i600", "C": 0.1, "max_iter": 600},
    {"model_name": "cfg_c0_3_i600", "C": 0.3, "max_iter": 600},
    {"model_name": "cfg_c1_0_i600", "C": 1.0, "max_iter": 600},
    {"model_name": "cfg_c3_0_i600", "C": 3.0, "max_iter": 600},
    {"model_name": "cfg_c10_i600", "C": 10.0, "max_iter": 600},
    {"model_name": "cfg_c1_0_i1400", "C": 1.0, "max_iter": 1400},
]


def train_eval_candidate(cfg: dict) -> dict:
    """
    Treina e avalia uma configuracao candidata do baseline PU.

    Objetivo estatistico:
    Medir, em amostra de tuning, o trade-off entre discriminacao em s,
    convergencia numerica e custo de treino.

    Logica metodologica:
    - A regressao logistica atua como classificador base do Elkan & Noto.
    - O pipeline inclui preprocessamento + wrapper PU.
    - A avaliacao usa predict_proba na validacao e extrai a coluna associada
      a classe positiva observada.
    - AUC em s e tratada como criterio proxy de qualidade do score.

    Saidas de auditoria:
    A funcao devolve um dicionario com:
    - identificacao da configuracao;
    - AUC de validacao;
    - tempo de treino;
    - warnings de convergencia;
    - numero maximo de iteracoes efetivamente usado;
    - indicador sintetico de convergencia;
    - estimativa de c;
    - origem das classes encontradas no modelo.
    """

    # -----------------------------------------------------
    # Classificador base: regressao logistica
    # -----------------------------------------------------
    # Objetivo:
    # Estimar uma fronteira probabilistica simples, estavel e interpretavel,
    # adequada ao baseline PU da Trilha B.
    #
    # Interpretacao:
    # C controla intensidade da regularizacao l2; max_iter controla o limite
    # de iteracoes permitido ao solver para atingir convergencia.

    estimator = LogisticRegression(
        solver="saga", 
        C=float(cfg["C"]),
        max_iter=int(cfg["max_iter"]),
        n_jobs=-1,
        random_state=SEED,
    )

    # -----------------------------------------------------
    # Wrapper PU de Elkan & Noto
    # -----------------------------------------------------
    # Objetivo:
    # Ajustar o classificador base ao cenario positivo-unlabeled, incluindo a
    # estimativa da constante c a partir de um hold-out de positivos.
    #
    # Interpretacao:
    # hold_out_ratio define a fracao de positivos reservada internamente para
    # estimar c = P(s=1 | y=1), parametro central do ajuste Elkan & Noto.

    pu_clf = pulearn.ElkanotoPuClassifier(
        estimator=estimator,
        hold_out_ratio=HOLD_OUT_RATIO_MAIN,
        random_state=SEED,
    )

    # -----------------------------------------------------
    # Pipeline: preprocessamento + PU classifier
    # -----------------------------------------------------
    # Objetivo:
    # Garantir que imputacao/escala sejam aprendidas apenas no conjunto de
    # treino da etapa atual, evitando vazamento de informacao.
    #
    # Interpretacao:
    # O pipeline centraliza a sequencia correta de transformacao e ajuste.

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("pu", pu_clf),
        ]
    )

    # -----------------------------------------------------
    # Treino com captura explicita de warnings
    # -----------------------------------------------------
    # Objetivo:
    # Registrar sinais de nao convergencia e medir custo computacional.
    #
    # Interpretacao:
    # Warnings de convergencia ou n_iter muito proximo de max_iter podem indicar
    # que a configuracao ainda esta numericamente instavel.

    with warnings.catch_warnings(record=True) as w_list:
        warnings.simplefilter("always")
        t0 = time.perf_counter()
        model.fit(X_train_tune, y_train_tune.to_numpy(dtype="int8", copy=False))
        train_seconds = time.perf_counter() - t0

    # -----------------------------------------------------
    # Identificacao robusta da classe positiva
    # -----------------------------------------------------
    # Objetivo:
    # Evitar assumir implicitamente que a probabilidade positiva esta sempre na
    # coluna 1 de predict_proba.
    #
    # Interpretacao:
    # Esse cuidado e importante porque wrappers e estimadores internos podem
    # alterar a ordem de classes na saida probabilistica.

    pu_step = model.named_steps["pu"]
    pos_idx, class_source, class_labels = get_positive_class_index(pu_step, positive_label=1)

    # -----------------------------------------------------
    # Score de validacao e AUC proxy em s
    # -----------------------------------------------------
    # Objetivo:
    # Avaliar a capacidade da configuracao de ordenar s=1 acima de s=0 na
    # amostra de validacao usada no tuning.
    #
    # Interpretacao:
    # AUC alta sugere melhor separacao observacional em s, o que tende a apoiar
    # melhor qualidade do score PU no baseline.

    proba_val = np.asarray(model.predict_proba(X_val_tune))
    if proba_val.ndim != 2 or proba_val.shape[1] <= pos_idx:
        raise ValueError(f"Saida inesperada no tuning: shape={proba_val.shape}, pos_idx={pos_idx}")

    score_val = proba_val[:, pos_idx]
    auc_val = roc_auc_score(y_val_tune, score_val)

    # -----------------------------------------------------
    # Diagnostico de convergencia
    # -----------------------------------------------------
    # Objetivo:
    # Consolidar evidencias de estabilidade numerica do estimador.
    #
    # Interpretacao:
    # - convergence_warnings > 0 sugere treino incompleto ou instavel;
    # - n_iter_max muito proximo de max_iter sugere solver no limite;
    # - converged resume essas informacoes para comparacao rapida.

    conv_warn_count = sum(1 for w in w_list if issubclass(w.category, ConvergenceWarning))

    if hasattr(pu_step, "estimator_") and hasattr(pu_step.estimator_, "n_iter_"):
        n_iter_max = float(np.max(pu_step.estimator_.n_iter_))
    else:
        n_iter_max = np.nan

    converged = bool((conv_warn_count == 0) and (np.isnan(n_iter_max) or (n_iter_max < cfg["max_iter"])))

    # -----------------------------------------------------
    # Registro da constante c estimada
    # -----------------------------------------------------
    # Objetivo:
    # Guardar a estimativa interna de c produzida pelo wrapper PU.
    #
    # Interpretacao:
    # Variacoes fortes de c entre configuracoes podem indicar sensibilidade do
    # baseline ao ajuste probabilistico interno.

    c_est = float(getattr(pu_step, "c")) if hasattr(pu_step, "c") else np.nan

    return {
        "model_name": cfg["model_name"],
        "stage": "tuning_sample",
        "C": float(cfg["C"]),
        "max_iter": int(cfg["max_iter"]),
        "auc_val": float(auc_val),
        "train_seconds": float(train_seconds),
        "convergence_warnings": int(conv_warn_count),
        "n_iter_max": float(n_iter_max),
        "converged": converged,
        "c_est": c_est,
        "class_source": class_source,
        "class_labels": str(class_labels),
        "n_train_used": int(len(X_train_tune)),
        "n_val_used": int(len(X_val_tune)),
    }


# ---------------------------------------------------------
# Execucao do tuning controlado
# ---------------------------------------------------------
# Objetivo:
# Treinar todas as configuracoes candidatas e consolidar os resultados em um
# unico dataframe comparativo.
#
# Interpretacao:
# Essa etapa produz uma visao sintetica da relacao entre desempenho,
# convergencia e custo computacional.

rows = []
print("Iniciando tuning controlado (amostra train/val)...")
for cfg in candidate_configs:
    print(f"- Treinando {cfg['model_name']} | C={cfg['C']} | max_iter={cfg['max_iter']}")
    rows.append(train_eval_candidate(cfg))

df_model_compare = pd.DataFrame(rows)

# ---------------------------------------------------------
# Ordenacao do quadro comparativo
# ---------------------------------------------------------
# Objetivo:
# Priorizar configuracoes com maior AUC em validacao, preferindo aquelas com
# convergencia adequada e menor tempo de treino.
#
# Interpretacao:
# A ordenacao nao substitui julgamento metodologico, mas organiza a evidencia
# para escolha mais transparente da configuracao final.

df_model_compare = df_model_compare.sort_values(
    by=["auc_val", "converged", "train_seconds"],
    ascending=[False, False, True],
).reset_index(drop=True)

# ---------------------------------------------------------
# Snapshot explicito do baseline original
# ---------------------------------------------------------
# Objetivo:
# Destacar a configuracao inicial congelada dentro do quadro comparativo.
#
# Interpretacao:
# Isso facilita demonstrar, no relatorio ou artigo, se houve ganho real ou
# apenas manutencao da configuracao baseline.

baseline_snapshot_df = df_model_compare[df_model_compare["model_name"] == "baseline_original"].copy()

print("\nResultado do tuning (ordenado):")
display(df_model_compare)

print("\nBaseline congelado para comparacao:")
display(baseline_snapshot_df)

### Interpretacao parcial da Secao 6 (congelamento e tuning controlado)

**Contexto do bloco**
- Primeiro, o baseline original e congelado para servir de referencia comparavel.
- Depois, sao testadas configuracoes de `C` e `max_iter` em amostra de treino/validacao.

**Objetivo estatistico**
- Buscar configuracao com melhor separacao proxy em validacao (`AUC sobre s`) com convergencia estavel e custo computacional controlado.

**Como interpretar `df_model_compare`**
- `auc_val` maior e desejavel (proxy de separacao de `s`).
- `converged=True` e baixa contagem de `ConvergenceWarning` indicam ajuste numerico mais confiavel.
- `c_est` resume a taxa estimada de selecao dos positivos no mecanismo Elkan-Noto.
- Como `hold_out_ratio` foi mantido fixo, a comparacao aqui isola principalmente sensibilidade a regularizacao (`C`) e orcamento de iteracoes (`max_iter`).


### Regra metodologica adotada para a escolha do modelo final

Nesta trilha, a regra principal continua sendo escolher o modelo final **preferencialmente entre configuracoes com `converged=True`**.

A motivacao e tratar convergencia como criterio minimo de sanidade numerica. Ganhos marginais de AUC proxy nao justificam promover automaticamente uma configuracao que ainda emite `ConvergenceWarning` no tuning.

**Excecao formal prevista para o modo SMOKE ou para cenarios numericamente mais rigidos:**
- se nenhuma configuracao convergir, a trilha nao para automaticamente;
- nessa situacao, o notebook escolhe a configuracao com menor numero de `ConvergenceWarning` e, dentro desse subconjunto, usa `auc_val` e `train_seconds` como desempate;
- a excecao fica registrada em output e no manifest para auditoria metodologica.


In [ ]:
# ---------------------------------------------------------
# Re-treino final com configuracao escolhida na etapa de tuning
# ---------------------------------------------------------
# Contexto:
# A etapa anterior comparou poucas configuracoes da regressao logistica
# dentro do wrapper PU de Elkan & Noto, usando uma amostra controlada
# de train/val para decidir uma configuracao suficientemente boa,
# estavel e reprodutivel.
#
# Objetivo estatistico:
# Reajustar o modelo final no conjunto completo de treino, agora com a
# configuracao escolhida, para obter a versao oficial da Trilha B.
#
# Logica metodologica:
# - O tuning foi apenas uma etapa de selecao operacional.
# - O modelo final deve ser treinado no treino completo, e nao apenas na
#   amostra de tuning, para aproveitar toda a informacao disponivel.
# - Nesta fase, o foco principal e garantir:
#   1) configuracao final explicitamente registrada;
#   2) treino reprodutivel;
#   3) ausencia de sinais fortes de nao convergencia;
#   4) mapeamento correto da classe positiva em predict_proba;
#   5) registro da estimativa interna de c.
#
# Interpretacao:
# O score final da Trilha B depende criticamente de dois elementos:
# - a ordem correta das classes internas do modelo;
# - a estimativa de c produzida pelo Elkan & Noto.
# Sem esse controle, ha risco de interpretar a coluna errada de
# predict_proba ou reportar scores com direcao invertida.

# ---------------------------------------------------------
# Selecao da melhor configuracao a partir do quadro comparativo
# ---------------------------------------------------------
# Objetivo:
# Recuperar, de forma programatica e auditavel, a melhor configuracao
# definida na etapa de tuning.
#
# Criterio pratico adotado:
# A configuracao ja chega ordenada em df_model_compare segundo desempenho
# proxy (AUC em s), convergencia e tempo de treino.
#
# Interpretacao:
# Este bloco formaliza a escolha da configuracao vencedora. Em artigo ou
# relatorio, isso ajuda a mostrar que a decisao nao foi arbitraria.
#
# Regra conservadora adotada:
# A escolha final considera apenas configuracoes com converged=True.
# Isso trata convergencia como criterio minimo de elegibilidade numerica
# antes de usar AUC proxy e tempo de treino como criterios de desempate.

df_model_compare_ok = df_model_compare[df_model_compare["converged"]].copy()
if len(df_model_compare_ok) > 0:
    best_selection_rule = "prefer_converged_true_then_auc_val_then_train_seconds"
    best_row = (
        df_model_compare_ok
        .sort_values(by=["auc_val", "train_seconds"], ascending=[False, True])
        .iloc[0]
        .to_dict()
    )
else:
    best_selection_rule = "fallback_min_convergence_warnings_then_auc_val_then_train_seconds"
    best_row = (
        df_model_compare
        .sort_values(by=["convergence_warnings", "auc_val", "train_seconds"], ascending=[True, False, True])
        .iloc[0]
        .to_dict()
    )

best_config = {
    "model_name": str(best_row["model_name"]),
    "C": float(best_row["C"]),
    "max_iter": int(best_row["max_iter"]),
}

print("Melhor configuracao selecionada:")
print(best_config)
print(f"- converged no tuning? {bool(best_row['converged'])}")
print(f"- convergence_warnings no tuning: {int(best_row['convergence_warnings'])}")
print(f"- candidatos convergidos elegiveis: {len(df_model_compare_ok)}")
print(f"- regra aplicada: {best_selection_rule}")

# ---------------------------------------------------------
# Cuidado metodologico importante
# ---------------------------------------------------------
# Observacao:
# Mesmo com a regra conservadora acima, o quadro comparativo completo
# continua util para mostrar a distancia entre configuracoes convergidas
# e nao convergidas, bem como o custo de insistir em maior agressividade
# de regularizacao/iteracoes por ganhos minimos de AUC proxy.
#
# Como AUC em s ja e uma metrica proxy, tratar convergencia como filtro
# de elegibilidade reduz o risco de superinterpretar diferencas minimas
# em um criterio que nao observa diretamente o y latente.

# ---------------------------------------------------------
# Definicao do estimador base final
# ---------------------------------------------------------
# Objetivo:
# Instanciar novamente a regressao logistica com os hiperparametros
# escolhidos na etapa de tuning.
#
# Interpretacao:
# - C controla o grau de regularizacao l2;
# - max_iter define o limite de iteracoes do solver;
# - random_state garante reprodutibilidade do ajuste;
# - n_jobs=-1 busca explorar paralelismo, quando suportado.

base_estimator_final = LogisticRegression(
    solver="saga",
    C=best_config["C"],
    max_iter=best_config["max_iter"],
    n_jobs=-1,
    random_state=SEED,
)

# ---------------------------------------------------------
# Wrapper PU final de Elkan & Noto
# ---------------------------------------------------------
# Objetivo:
# Envolver o estimador base no procedimento PU sob hipotese SCAR.
#
# Logica metodologica:
# O Elkan & Noto treina o classificador base para aproximar P(s=1 | x)
# e estima internamente a constante c = P(s=1 | y=1), usando uma fracao
# dos positivos rotulados definida por hold_out_ratio.
#
# Interpretacao:
# O predict_proba do wrapper devolve probabilidades ajustadas para o
# alvo latente y, e nao apenas para o rotulo observado s.
#
# Observacao:
# hold_out_ratio=HOLD_OUT_RATIO_MAIN significa que 10% dos positivos rotulados sao
# reservados internamente para estimar c. Esse valor e um parametro do
# metodo, nao a proporcao de s=1 na base.

pu_classifier_final = pulearn.ElkanotoPuClassifier(
    estimator=base_estimator_final,
    hold_out_ratio=HOLD_OUT_RATIO_MAIN,
    random_state=SEED,
)

# ---------------------------------------------------------
# Pipeline final: preprocessamento + modelo PU
# ---------------------------------------------------------
# Objetivo:
# Garantir que imputacao, escala e ajuste do modelo ocorram de forma
# encadeada e sem vazamento de informacao.
#
# Interpretacao:
# O preprocessador e aprendido exclusivamente sobre X_train, e o modelo
# PU recebe as features ja tratadas na ordem correta.

model_final = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("pu", pu_classifier_final),
    ]
)

# ---------------------------------------------------------
# Treino final no conjunto completo de treino
# ---------------------------------------------------------
# Objetivo:
# Ajustar a versao definitiva da Trilha B no treino integral.
#
# Justificativa:
# A etapa de tuning usou subamostra apenas para comparacao rapida entre
# configuracoes. O treino final deve aproveitar todo o treino disponivel.
#
# Captura de warnings:
# warnings.catch_warnings(record=True) permite registrar warnings de
# convergencia e outros avisos relevantes para auditoria.
#
# Conversao de y:
# O vetor y_train e convertido para numpy int8 por eficiencia de memoria
# e por compatibilidade operacional com o ajuste do modelo.

with warnings.catch_warnings(record=True) as w_final:
    warnings.simplefilter("always")
    y_train_np = y_train.to_numpy(dtype="int8", copy=False)

    t0 = time.perf_counter()
    model_final.fit(X_train, y_train_np)
    train_seconds_final = time.perf_counter() - t0

# ---------------------------------------------------------
# Acesso ao passo PU ja treinado
# ---------------------------------------------------------
# Objetivo:
# Recuperar o wrapper Elkan & Noto treinado dentro do pipeline, para
# inspecionar atributos internos relevantes.
#
# Interpretacao:
# E desse objeto que saem:
# - classes internas;
# - estimativa de c;
# - eventualmente informacoes do estimador interno.

pu_step_final = model_final.named_steps["pu"]

# ---------------------------------------------------------
# Mapeamento robusto da classe positiva
# ---------------------------------------------------------
# Objetivo:
# Identificar qual coluna de predict_proba corresponde a classe positiva
# (rotulada como 1), sem assumir implicitamente que ela esta sempre na
# coluna 1.
#
# Justificativa:
# Wrappers e estimadores podem alterar a ordem das classes. Em contexto
# PU, usar a coluna errada compromete completamente o ranking final.
#
# Saidas:
# - proba_class_index_for_1: indice correto da classe positiva;
# - class_source_final: de onde veio a informacao das classes;
# - class_labels_final: rotulos observados nas classes internas.

proba_class_index_for_1, class_source_final, class_labels_final = get_positive_class_index(
    pu_step_final,
    positive_label=1,
)

# ---------------------------------------------------------
# Diagnostico de convergencia do treino final
# ---------------------------------------------------------
# Objetivo:
# Consolidar sinais de estabilidade numerica apos o re-treino completo.
#
# Logica:
# - convergence_warnings_final conta warnings do tipo ConvergenceWarning;
# - n_iter_max_final procura o maior numero de iteracoes efetivamente
#   usado pelo estimador interno, quando disponivel;
# - converged_final resume essas informacoes em um indicador booleano.
#
# Observacao:
# Em alguns wrappers, n_iter_ pode nao estar facilmente exposto; nesse
# caso, o valor pode permanecer como NaN, e a leitura de convergencia
# depende mais fortemente dos warnings.

convergence_warnings_final = sum(
    1 for w in w_final if issubclass(w.category, ConvergenceWarning)
)

if hasattr(pu_step_final, "estimator_") and hasattr(pu_step_final.estimator_, "n_iter_"):
    n_iter_max_final = float(np.max(pu_step_final.estimator_.n_iter_))
else:
    n_iter_max_final = np.nan

converged_final = bool(
    (convergence_warnings_final == 0)
    and (np.isnan(n_iter_max_final) or (n_iter_max_final < best_config["max_iter"]))
)

# ---------------------------------------------------------
# Registro da constante c estimada no treino final
# ---------------------------------------------------------
# Objetivo:
# Guardar explicitamente a estimativa interna de c, produzida pelo
# Elkan & Noto no ajuste final.
#
# Interpretacao substantiva:
# c representa a probabilidade de um positivo verdadeiro aparecer como
# positivo rotulado sob a hipotese SCAR.
#
# Leitura:
# - c menor sugere maior subrotulagem dos positivos;
# - c maior sugere que parcela maior dos positivos verdadeiros foi
#   observada como s=1.
#
# Observacao:
# Essa nao e a mesma coisa que a proporcao observada de s=1 na base.

c_estimado = float(getattr(pu_step_final, "c")) if hasattr(pu_step_final, "c") else np.nan

# ---------------------------------------------------------
# Log final de auditoria do modelo oficial da Trilha B
# ---------------------------------------------------------
# Objetivo:
# Exibir os principais elementos necessarios para validacao manual do
# treino final e futura reproducao do experimento.
#
# Itens criticos:
# - tempo de treino;
# - origem e ordem das classes internas;
# - indice correto da classe positiva;
# - warnings de convergencia;
# - numero maximo de iteracoes observado;
# - status sintetico de convergencia;
# - valor final de c.
#
# Interpretacao:
# Esse bloco funciona como um checkpoint tecnico do baseline PU antes da
# geracao de scores e rankings em validacao/teste.

print("\nTreino final concluido.")
print(f"- train_seconds_final: {train_seconds_final:.2f}s")
print(f"- class_source_final: {class_source_final}")
print(f"- class_labels_final: {class_labels_final}")
print(f"- proba_class_index_for_1: {proba_class_index_for_1}")
print(f"- convergence_warnings_final: {convergence_warnings_final}")
print(f"- n_iter_max_final: {n_iter_max_final}")
print(f"- converged_final: {converged_final}")
print(f"- c_estimado: {c_estimado}")


In [ ]:
# ---------------------------------------------------------
# Avaliação do modelo final (val e test)
# ---------------------------------------------------------
# Objetivo:
# Calcular AUC utilizando os scores ajustados pelo PU Learning,
# garantindo uso correto da classe positiva.

from sklearn.metrics import roc_auc_score
import numpy as np

# ---------- VAL ----------
proba_val_final = np.asarray(model_final.predict_proba(X_val))

if proba_val_final.ndim != 2 or proba_val_final.shape[1] <= proba_class_index_for_1:
    raise ValueError(
        f"Saida inesperada em val: shape={proba_val_final.shape}, "
        f"pos_idx={proba_class_index_for_1}"
    )

score_val_final = proba_val_final[:, proba_class_index_for_1]
auc_val_final = roc_auc_score(y_val, score_val_final)

# ---------- TEST ----------
proba_test_final = np.asarray(model_final.predict_proba(X_test))

if proba_test_final.ndim != 2 or proba_test_final.shape[1] <= proba_class_index_for_1:
    raise ValueError(
        f"Saida inesperada em test: shape={proba_test_final.shape}, "
        f"pos_idx={proba_class_index_for_1}"
    )

score_test_final = proba_test_final[:, proba_class_index_for_1]
auc_test_final = roc_auc_score(y_test, score_test_final)

print("\nAvaliação do modelo final:")
print(f"- AUC (val):  {auc_val_final:.6f}")
print(f"- AUC (test): {auc_test_final:.6f}")

### Interpretacao parcial da Secao 6 (re-treino final)

**Objetivo do bloco**
- Reajustar o modelo final em todo o conjunto de treino com a melhor configuracao do tuning.

**Pontos de auditoria**
- Inspecao explicita do mapeamento de classes (`class_labels` e indice da classe positiva).
- Registro de tempo de treino, `c` estimado e sinais de convergencia.
- Registro explicito de se a configuracao escolhida havia ou nao convergido no tuning.

**Interpretacao esperada**
- O indice da classe positiva deve corresponder a `s=1`; isso garante direcao correta do score no `predict_proba`.
- Se a configuracao selecionada vier acompanhada de warning no tuning, isso deve ser lido como um ponto metodologico pendente, nao como detalhe operacional irrelevante.


In [ ]:
# ---------------------------------------------------------
# Consolidacao final da tabela comparativa de modelos
# ---------------------------------------------------------
# Contexto:
# Reune, em um unico artefato, os resultados obtidos em duas etapas:
# (i) candidatos avaliados no tuning controlado em amostra e
# (ii) modelo final re-treinado no conjunto completo de treino.
#
# Objetivo estatistico:
# Organizar evidencias de desempenho proxy, estabilidade numerica,
# custo computacional e parametros do modelo em uma tabela unica,
# adequada para auditoria metodologica e documentacao no artigo.
#
# Interpretacao:
# Esta tabela nao serve apenas para "mostrar quem ganhou".
# Ela documenta:
# - sensibilidade (ou insensibilidade) do baseline a hiperparametros;
# - consistencia entre tuning e treino final;
# - estabilidade da estimativa de c;
# - ausencia ou presenca de problemas de convergencia.
#
# Observacao importante:
# As linhas da etapa "tuning_sample" usam AUC em amostra de validacao
# do tuning, enquanto a linha "full_train" registra AUC em validacao e
# teste do modelo final. Portanto, as AUCs nao sao perfeitamente
# comparaveis entre todas as linhas, mas convivem utilmente na mesma
# tabela para fins de rastreabilidade.

# ---------------------------------------------------------
# Padronizacao de schema para concatenacao
# ---------------------------------------------------------
# Objetivo:
# Garantir que o dataframe do tuning e a linha do treino final compartilhem
# as mesmas colunas, inclusive auc_test, que nao existia originalmente
# nas linhas do tuning.
#
# Interpretacao:
# Isso evita inconsistencias de schema e facilita exportacao posterior
# para CSV, Excel ou uso em tabelas do artigo.

df_model_compare_for_concat = df_model_compare.copy()

if "auc_test" not in df_model_compare_for_concat.columns:
    df_model_compare_for_concat["auc_test"] = np.nan

# ---------------------------------------------------------
# Linha-resumo do modelo final treinado em full train
# ---------------------------------------------------------
# Objetivo:
# Registrar, em formato compativel com a tabela comparativa, os principais
# resultados do modelo oficial da Trilha B.
#
# Itens incluidos:
# - hiperparametros finais;
# - AUC em validacao e teste;
# - tempo de treino;
# - diagnostico de convergencia;
# - estimativa final de c;
# - origem e ordem das classes;
# - volume de dados efetivamente utilizado.
#
# Interpretacao:
# Esta linha sintetiza a configuracao que, de fato, sera usada para gerar
# scores, rankings e analises substantivas do estudo.

final_train_row = {
    "model_name": "best_full_train",
    "stage": "full_train",
    "C": float(best_config["C"]),
    "max_iter": int(best_config["max_iter"]),
    "auc_val": float(auc_val_final),
    "auc_test": float(auc_test_final),
    "train_seconds": float(train_seconds_final),
    "convergence_warnings": int(convergence_warnings_final),
    "n_iter_max": float(n_iter_max_final),
    "converged": bool(converged_final),
    "c_est": float(c_estimado) if pd.notna(c_estimado) else np.nan,
    "class_source": str(class_source_final),
    "class_labels": str(class_labels_final),
    "n_train_used": int(len(X_train)),
    "n_val_used": int(len(X_val)),
}

# ---------------------------------------------------------
# Concatenacao da tabela comparativa final
# ---------------------------------------------------------
# Objetivo:
# Empilhar resultados do tuning e do treino final em um unico dataframe.
#
# Interpretacao:
# O resultado final preserva a trilha completa de decisao metodologica,
# do teste rapido em amostra ate o modelo oficial treinado no conjunto
# completo de treino.

df_model_compare_final = pd.concat(
    [df_model_compare_for_concat, pd.DataFrame([final_train_row])],
    ignore_index=True,
)

# ---------------------------------------------------------
# Ordenacao opcional para leitura analitica
# ---------------------------------------------------------
# Objetivo:
# Melhorar legibilidade da tabela final, colocando primeiro as etapas de
# tuning e, ao final, o modelo oficial em full train.
#
# Interpretacao:
# A ordenacao abaixo preserva uma leitura natural do fluxo experimental:
# candidatos primeiro, modelo final depois.

stage_order = {"tuning_sample": 0, "full_train": 1}
df_model_compare_final["_stage_order"] = df_model_compare_final["stage"].map(stage_order).fillna(99)

df_model_compare_final = (
    df_model_compare_final
    .sort_values(by=["_stage_order", "auc_val", "converged", "train_seconds"],
                 ascending=[True, False, False, True])
    .drop(columns="_stage_order")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Exibicao da tabela consolidada
# ---------------------------------------------------------
# Objetivo:
# Mostrar, em um unico quadro, os resultados relevantes para
# documentacao do baseline PU.
#
# Leitura recomendada:
# - tuning_sample: comparar sensibilidade do baseline a C e max_iter;
# - full_train: verificar desempenho final, estabilidade e valor de c.
#
# Observacao:
# Para discussao substantiva do trabalho, a linha "best_full_train" deve
# ser tratada como a referencia principal da Trilha B.

print("Resumo comparativo final (candidatos do tuning + modelo final):")
display(df_model_compare_final)

In [ ]:
# ---------------------------------------------------------
# Diagnostico complementar de convergencia do estimador final
# ---------------------------------------------------------
# Contexto:
# Inspecao direta de n_iter_ no estimador interno, quando esse atributo
# estiver exposto pelo wrapper PU.
#
# Objetivo estatistico:
# Verificar se o solver utilizou numero de iteracoes proximo ao limite
# max_iter no treino final.
#
# Interpretacao:
# Este bloco e complementar. A ausencia de n_iter_ nao invalida o modelo
# final quando nao ha ConvergenceWarning e o treino foi concluido com
# sucesso.

if hasattr(pu_step_final, "estimator_") and hasattr(pu_step_final.estimator_, "n_iter_"):
    print("Iteracoes do solver no modelo final:", pu_step_final.estimator_.n_iter_)
else:
    print("n_iter_ nao disponivel no estimador interno do modelo final.")

### Diagnosticos minimos de modelagem (Secao 6)

Este notebook registra os diagnosticos minimos obrigatorios para modelos:
- verificacao de convergencia (`n_iter_` e warnings),
- inspecao de classes internas do modelo,
- estatisticas basicas de treino e custo computacional.

Se o solver encostar no limite de iteracoes (`n_iter_` proximo de `max_iter`) ou gerar warnings recorrentes, a recomendacao e revisar escala, regularizacao (`C`) e limite de iteracoes em nova rodada controlada.

Observacao adicional:
- quando `n_iter_` nao fica exposto pelo wrapper, os warnings de convergencia passam a ser o principal sinal operacional de estabilidade numerica;
- por isso, o notebook registra simultaneamente `convergence_warnings`, `converged` e o valor de `c` para auditoria integrada.


## 7) Predições e outputs (scores e ranking)

Geramos `score_pu` no teste com `predict_proba(X_test)` usando mapeamento explícito da classe positiva (`s=1`).

Não fazemos inversão heurística baseada em `s` do teste. A direção do score é definida apenas pelo mapeamento de classes do modelo.


In [ ]:
# ---------------------------------------------------------
# Geracao de scores finais e ranking operacional no conjunto de teste
# ---------------------------------------------------------
# Contexto:
# Aplicacao do modelo final da Trilha B ao conjunto de teste para gerar
# scores continuos ajustados pelo metodo PU de Elkan & Noto.
#
# Objetivo estatistico:
# Produzir uma base auditavel de scores fora da amostra e organizar o
# ranking dos casos unlabeled (s=0), preservando o mapeamento correto da
# classe positiva no predict_proba.
#
# Logica metodologica:
# O score extraido corresponde a coluna associada a classe positiva
# identificada explicitamente no modelo final. Isso evita erro de
# interpretacao por inversao da ordem das classes internas.
#
# Interpretacao:
# - s=1 + score alto: acesso observado e coerente com o padrao aprendido;
# - s=0 + score alto: demanda latente / candidatos potenciais ao credito;
# - s=0 + score baixo: perfis distantes do padrao observado de acesso,
#   compatveis com a hipotese de exclusao estrutural.
#
# Observacao:
# No Elkan & Noto, o predict_proba ja incorpora o ajuste por c. Portanto,
# o score abaixo deve ser lido como estimativa associada ao alvo latente y.
# Ainda assim, nesta trilha ele e usado principalmente como instrumento de
# ordenacao operacional, e nao como probabilidade calibrada.

# ---------------------------------------------------------
# Predicao probabilistica no conjunto de teste
# ---------------------------------------------------------
# Objetivo:
# Gerar probabilidades ajustadas pelo modelo final para cada observacao
# do conjunto de teste.
#
# Interpretacao:
# A matriz proba_test deve ter duas colunas, e a coluna da classe positiva
# sera acessada usando o indice explicitamente mapeado em
# proba_class_index_for_1.

proba_test = np.asarray(model_final.predict_proba(X_test))

if proba_test.ndim != 2 or proba_test.shape[1] <= proba_class_index_for_1:
    raise ValueError(
        f"Saida inesperada de predict_proba no teste: shape={proba_test.shape}, "
        f"pos_idx={proba_class_index_for_1}"
    )

# ---------------------------------------------------------
# Extracao do score bruto da classe positiva
# ---------------------------------------------------------
# Objetivo:
# Recuperar o score associado a classe positiva (1) sem assumir
# implicitamente que ela esta sempre na coluna 1.
#
# Interpretacao:
# score_pu_raw representa o score oficial da Trilha B para ranking
# operacional e avaliacao proxy. Nao o tratamos como probabilidade
# calibrada; sua funcao principal e ordenar casos.

score_pu_raw = proba_test[:, proba_class_index_for_1].astype("float64")

# ---------------------------------------------------------
# Higienizacao numerica do score
# ---------------------------------------------------------
# Objetivo:
# Tratar ocorrencias numericas anormais (NaN, +inf, -inf) que poderiam
# comprometer ordenacao, estatisticas e exportacao.
#
# Interpretacao:
# - NaN e -inf sao enviados para 0.0;
# - +inf e truncado no maior float64 disponivel.
# Isso preserva rastreabilidade do score bruto sem interromper o fluxo.

score_pu_raw = np.nan_to_num(
    score_pu_raw,
    nan=0.0,
    posinf=np.finfo("float64").max,
    neginf=0.0,
)

# ---------------------------------------------------------
# Versao clipped do score para analise descritiva
# ---------------------------------------------------------
# Objetivo:
# Produzir uma versao limitada a [0, 1], util apenas para tabelas
# descritivas, graficos e revisoes auxiliares.
#
# Interpretacao:
# Em Elkan & Noto, a correcao por c pode gerar valores fora de [0,1]
# antes de qualquer truncamento numerico. Por isso, preservamos:
# - score_pu_raw: score oficial da Trilha B para ranking e avaliacao;
# - score_pu_clipped: apoio descritivo e visual, sem substituir o raw.

score_pu_clipped = np.clip(score_pu_raw, 0.0, 1.0).astype("float32")

# ---------------------------------------------------------
# Montagem da base de scores no teste
# ---------------------------------------------------------
# Objetivo:
# Consolidar identificadores, metadados e scores em uma base unica para
# analise fora da amostra.
#
# Interpretacao:
# Incluimos:
# - indice_linha_original: rastreabilidade com a base de origem;
# - cpf_agricultor: unidade analitica principal no nivel pessoa;
# - id_familia: suporte a analises por agrupamento familiar;
# - ibge: metadado territorial para agregacoes futuras;
# - s: rotulo observado;
# - score_pu_raw: score oficial da Trilha B para ranking e avaliacao;
# - score_pu_clipped: coluna auxiliar para leitura descritiva.

meta_cols_scores = ["cpf_agricultor", "id_familia", "ibge", TARGET_COL]
missing_meta_cols = [c for c in meta_cols_scores if c not in df.columns]
if missing_meta_cols:
    raise ValueError(f"Colunas ausentes para base de scores: {missing_meta_cols}")

df_scores_test = (
    df.iloc[idx_test][meta_cols_scores]
    .copy()
    .rename(columns={TARGET_COL: "s"})
)

df_scores_test["indice_linha_original"] = df.iloc[idx_test].index.to_numpy()
df_scores_test = df_scores_test.reset_index(drop=True)

df_scores_test["score_pu_raw"] = score_pu_raw.astype("float64")
df_scores_test["score_pu_clipped"] = score_pu_clipped

print("Shape do dataframe de scores (test):", df_scores_test.shape)
print("Colunas de saida:", df_scores_test.columns.tolist())

# ---------------------------------------------------------
# Estatisticas descritivas do score
# ---------------------------------------------------------
# Objetivo:
# Resumir a distribuicao do score no conjunto de teste.
#
# Interpretacao:
# O foco principal e caracterizar o score oficial (`score_pu_raw`),
# mantendo a versao clipped apenas como apoio descritivo por grupo.

score_stats_raw = df_scores_test["score_pu_raw"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99]
)
score_stats_clipped = df_scores_test["score_pu_clipped"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99]
)

print("\nEstatisticas de score_pu_raw:")
display(score_stats_raw.to_frame(name="score_pu_raw"))

print("\nEstatisticas de score_pu_clipped:")
display(score_stats_clipped.to_frame(name="score_pu_clipped"))

# ---------------------------------------------------------
# Diagnostico de scores fora de [0,1]
# ---------------------------------------------------------
# Objetivo:
# Verificar se a versao bruta do score apresentou valores fora da faixa
# probabilistica padrao.
#
# Interpretacao:
# Esse diagnostico e especialmente util em PU Learning com ajuste por c,
# onde a transformacao pode produzir valores acima de 1 em alguns casos.

outside_01 = int(((df_scores_test["score_pu_raw"] < 0) | (df_scores_test["score_pu_raw"] > 1)).sum())
pct_outside_01 = outside_01 / len(df_scores_test) if len(df_scores_test) > 0 else np.nan

print(f"\nScores brutos fora de [0,1]: {outside_01:,} ({pct_outside_01:.4%})")
if outside_01 > 0:
    print("Aviso: score_pu_raw segue como score oficial de ranking; score_pu_clipped fica restrito a leitura descritiva.")

# ---------------------------------------------------------
# Ranking operacional de unlabeled (s=0)
# ---------------------------------------------------------
# Objetivo:
# Ordenar os casos unlabeled no conjunto de teste segundo o score PU,
# priorizando agricultores com maior score estimado.
#
# Interpretacao:
# Este ranking materializa o principal produto operacional da Trilha B:
# identificar, entre os nao observados como positivos, os perfis mais
# semelhantes ao padrao de acesso aprendido pelo modelo.

ranking_unlabeled = (
    df_scores_test[df_scores_test["s"] == 0]
    .sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True])
    .reset_index(drop=True)
)

ranking_unlabeled_top1000 = ranking_unlabeled.head(1000).copy()

print(f"\nTotal unlabeled no teste (s=0): {len(ranking_unlabeled):,}")
print(f"Top 1000 unlabeled para inspecao: {ranking_unlabeled_top1000.shape}")

print("\nAmostra top 10 do ranking unlabeled (CPF mascarado):")
display(
    mask_cpf_frame(
        ranking_unlabeled_top1000[
            ["cpf_agricultor", "id_familia", "ibge", "s", "score_pu_raw", "score_pu_clipped"]
        ].head(10),
        columns=["cpf_agricultor"],
    )
)

# ---------------------------------------------------------
# Leitura por faixas de score entre os unlabeled
# ---------------------------------------------------------
# Objetivo:
# Produzir uma visao mais substantiva da distribuicao dos s=0 por faixas
# de score, indo alem do top 1000.
#
# Interpretacao:
# Essas faixas ajudam a separar:
# - demanda latente alta;
# - zona intermediaria de incerteza;
# - perfis de baixa aderencia ao padrao observado de acesso.

bins = [-np.inf, 0.20, 0.50, 0.80, np.inf]
labels = ["baixo_ate_0_20", "medio_0_20_0_50", "alto_0_50_0_80", "muito_alto_acima_0_80"]

ranking_unlabeled["faixa_score"] = pd.cut(
    ranking_unlabeled["score_pu_raw"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True,
)

resumo_faixas_unlabeled = (
    ranking_unlabeled["faixa_score"]
    .value_counts(dropna=False)
    .rename_axis("faixa_score")
    .reset_index(name="n")
)

resumo_faixas_unlabeled["pct"] = resumo_faixas_unlabeled["n"] / len(ranking_unlabeled)

print("\nDistribuicao dos unlabeled por faixa de score:")
display(resumo_faixas_unlabeled)

### Interpretacao da Secao 7 (scores e ranking)

**Objetivo analitico**
- Converter a saida do modelo em score continuo (`score_pu_raw`) para priorizacao operacional de casos.

**Decisao critica implementada**
- `score_pu_raw` e o score oficial da Trilha B para ranking e avaliacao proxy.
- `score_pu_clipped` permanece apenas como apoio descritivo, visual e exploratorio.
- Nao ha inversao de score baseada no `s` do teste.
- A direcao do score e determinada somente pelo mapeamento da classe positiva no modelo treinado.

**Como interpretar o ranking**
- O ranking em `s==0` ordena unlabeled por maior `score_pu_raw`.
- Esse ranking e o principal produto operacional da Trilha B.
- O `top 1000` e uma amostra operacional para inspecao qualitativa rapida.


## 8) Avaliacao rapida do baseline

Como nao temos negativos rotulados (`y=0`) observaveis, usamos metricas **proxy** no teste.
Todas as metricas principais desta secao sao calculadas a partir de `score_pu_raw`, que e o score oficial da Trilha B para ranking operacional.

1. AUC para separar `s`.
2. Recall@k sobre `s==1` para `k` em 1%, 5% e 10%.
3. Precision@k para qualificar a pureza do topo do ranking.

Tambem consolidamos um dataframe de comparacao entre modelos candidatos e modelo final.


In [ ]:
# ---------------------------------------------------------
# Avaliacao proxy no teste (AUC em s, Recall@k e Precision@k)
# ---------------------------------------------------------
# Contexto:
# Como y real nao e observado para todos os casos, utilizamos metricas proxy
# baseadas no rotulo observado s.
#
# Objetivo estatistico:
# - AUC em s: mede separacao entre positivos rotulados e unlabeled;
# - Recall@k: mede cobertura dos positivos no topo do ranking;
# - Precision@k: mede pureza do topo (proporcao de s=1).
#
# O que estas metricas NAO medem:
# Nao medem desempenho direto em y real. Os resultados devem ser interpretados
# como evidencia operacional (ranking e priorizacao), nao como validacao causal.

# ---------------------------------------------------------
# Preparacao dos vetores de avaliacao
# ---------------------------------------------------------
# Objetivo:
# Garantir alinhamento entre labels e scores e selecionar explicitamente o
# score correto (raw) para avaliacao.

y_test_eval = pd.Series(y_test).astype("int8").reset_index(drop=True)

score_eval = pd.to_numeric(
    df_scores_test["score_pu_raw"],  # score oficial da Trilha B
    errors="coerce"
).fillna(0.0).reset_index(drop=True)

if len(y_test_eval) != len(score_eval):
    raise ValueError(
        f"Tamanho inconsistente entre y_test ({len(y_test_eval)}) "
        f"e score_pu_raw ({len(score_eval)})."
    )

# ---------------------------------------------------------
# AUC (proxy em s)
# ---------------------------------------------------------
# Interpretacao:
# Mede capacidade do modelo de ordenar s=1 acima de s=0.

auc_s = roc_auc_score(y_test_eval, score_eval)

# ---------------------------------------------------------
# Preparacao para metricas de ranking
# ---------------------------------------------------------
score_np = score_eval.to_numpy(dtype="float64", copy=False)
y_np = y_test_eval.to_numpy(dtype="int8", copy=False)

ord_desc = np.argsort(-score_np)

positivos_totais = int((y_np == 1).sum())

if positivos_totais == 0:
    raise ValueError("Nao ha positivos no conjunto de teste para avaliar recall.")

k_fracoes = [0.01, 0.05, 0.10]
metricas_ranking = []

# ---------------------------------------------------------
# Calculo de Recall@k e Precision@k
# ---------------------------------------------------------
for frac in k_fracoes:
    k = int(np.ceil(frac * len(score_np)))
    k = max(k, 1)

    topk_idx = ord_desc[:k]

    positivos_topk = int((y_np[topk_idx] == 1).sum())

    recall_k = positivos_topk / positivos_totais
    precision_k = positivos_topk / k

    metricas_ranking.append(
        {
            "k_frac": frac,
            "k_absoluto": k,
            "positivos_topk": positivos_topk,
            "positivos_totais_test": positivos_totais,
            "recall_at_k": recall_k,
            "precision_at_k": precision_k,
        }
    )

df_ranking_k = pd.DataFrame(metricas_ranking)

# ---------------------------------------------------------
# Exibicao das metricas
# ---------------------------------------------------------
print("Metricas proxy no conjunto de teste:")
print(f"- AUC para separar s: {auc_s:.6f}")

print("\nMetricas de ranking (topo do score):")
display(df_ranking_k)

# ---------------------------------------------------------
# Consolidacao na tabela comparativa de modelos
# ---------------------------------------------------------
# Objetivo:
# Registrar desempenho final do modelo baseline para comparacao com outras
# trilhas (ex: SAR, nnPU, calibracoes, etc).

final_eval_row = {
    "model_name": "best_full_train_test_eval",
    "stage": "test_eval",
    "C": float(best_config["C"]),
    "max_iter": int(best_config["max_iter"]),
    "auc_val": np.nan,
    "train_seconds": float(train_seconds_final),
    "convergence_warnings": int(convergence_warnings_final),
    "n_iter_max": float(n_iter_max_final),
    "converged": bool(converged_final),
    "c_est": float(c_estimado) if pd.notna(c_estimado) else np.nan,
    "class_source": str(class_source_final),
    "class_labels": str(class_labels_final),
    "n_train_used": int(len(X_train)),
    "n_val_used": int(len(X_val)),

    # Metricas proxy
    "auc_test_proxy_s": float(auc_s),
    "recall_at_1pct": float(df_ranking_k.loc[df_ranking_k["k_frac"] == 0.01, "recall_at_k"].iloc[0]),
    "recall_at_5pct": float(df_ranking_k.loc[df_ranking_k["k_frac"] == 0.05, "recall_at_k"].iloc[0]),
    "recall_at_10pct": float(df_ranking_k.loc[df_ranking_k["k_frac"] == 0.10, "recall_at_k"].iloc[0]),
    "precision_at_1pct": float(df_ranking_k.loc[df_ranking_k["k_frac"] == 0.01, "precision_at_k"].iloc[0]),
    "precision_at_5pct": float(df_ranking_k.loc[df_ranking_k["k_frac"] == 0.05, "precision_at_k"].iloc[0]),
    "precision_at_10pct": float(df_ranking_k.loc[df_ranking_k["k_frac"] == 0.10, "precision_at_k"].iloc[0]),
}

df_model_compare_final = pd.concat(
    [df_model_compare_final, pd.DataFrame([final_eval_row])],
    ignore_index=True,
)

print("\nComparacao de modelos (com metricas de teste do final):")
display(df_model_compare_final)

### Interpretacao da Secao 8 (metricas proxy)

**Importante sobre escopo das metricas**
- AUC, Recall@k e Precision@k desta secao usam `score_pu_raw`.
- AUC foi calculada sobre `s` (rotulo observado), nao sobre `y` real nao observado.
- Portanto, esta AUC mede capacidade de separar **rotulados positivos** de **unlabeled**, sendo apenas proxy de desempenho PU.

**Leitura operacional**
- `Recall@k` em `s==1` indica concentracao de rotulados no topo do ranking definido por `score_pu_raw`.
- `Precision@k` descreve a pureza desse topo sob o rotulo observado `s`.
- `score_pu_clipped` nao substitui essas metricas; ele fica restrito a visualizacao e ao apoio descritivo.


In [ ]:
# ---------------------------------------------------------
# Diagnostico visual: distribuicao do score por s no teste
# ---------------------------------------------------------
# Contexto:
# Comparacao visual entre a distribuicao do score final para os grupos
# s=0 e s=1 no conjunto de teste.
#
# Objetivo estatistico:
# Verificar se os positivos rotulados (s=1) tendem, em media, a ocupar
# faixas mais altas de score que os unlabeled (s=0).
#
# Interpretacao:
# - Separacao visivel entre as distribuicoes sugere que o baseline captou
#   sinal util para ordenacao.
# - Sobreposicao parcial e esperada em PU Learning.
# - Sobreposicao excessiva sugere limite discriminativo do baseline e
#   reforca a necessidade de refinamento em trilhas seguintes.
#
# Observacao metodologica:
# Este grafico nao redefine o score oficial da trilha. Como score_pu_raw
# pode assumir valores acima de 1 apos o ajuste por c, utilizamos
# score_pu_clipped apenas para visualizacao descritiva, preservando
# score_pu_raw como base do ranking operacional e da avaliacao proxy.

import matplotlib.pyplot as plt

if "df_scores_test" not in globals():
    raise ValueError("df_scores_test nao encontrado. Gere a base de scores antes do diagnostico visual.")

required_plot_cols = {"s", "score_pu_clipped"}
missing_plot_cols = sorted(required_plot_cols - set(df_scores_test.columns))
if missing_plot_cols:
    raise ValueError(f"Colunas obrigatorias ausentes para o grafico: {missing_plot_cols}")

plt.figure(figsize=(8, 4))

plt.hist(
    df_scores_test.loc[df_scores_test["s"] == 0, "score_pu_clipped"],
    bins=50,
    alpha=0.6,
    density=True,
    label="s=0"
)

plt.hist(
    df_scores_test.loc[df_scores_test["s"] == 1, "score_pu_clipped"],
    bins=50,
    alpha=0.6,
    density=True,
    label="s=1"
)

plt.title("Distribuicao de score_pu_clipped por s (teste)")
plt.xlabel("score_pu_clipped")
plt.ylabel("densidade")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Resumo descritivo do score por grupo observado s
# ---------------------------------------------------------
# Contexto:
# Tabela-resumo para complementar o diagnostico visual das distribuicoes.
#
# Objetivo estatistico:
# Comparar tendencia central e dispersao do score entre s=0 e s=1.
#
# Interpretacao:
# Espera-se que s=1 apresente, em media, scores mais altos que s=0, ainda
# que exista sobreposicao entre os grupos. Esta leitura permanece apenas
# no plano descritivo; ranking e metricas principais seguem no raw.

score_group_summary = (
    df_scores_test
    .groupby("s")["score_pu_clipped"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)

print("Resumo descritivo de score_pu_clipped por s:")
display(score_group_summary)

In [ ]:
# ---------------------------------------------------------
# Nota de conformidade visual do projeto
# ---------------------------------------------------------
# Contexto:
# O projeto padroniza visualizacoes sem seaborn na Trilha B.
#
# Objetivo:
# Registrar explicitamente a opcao por matplotlib puro nesta etapa,
# mantendo o baseline com menos dependencias e maior previsibilidade
# de reproducao.

print("Visualizacao concluida com matplotlib (sem uso de seaborn na Trilha B).")

### Interpretacao do diagnostico de distribuicao de score

O histograma por `s` ajuda a verificar separacao basica das distribuicoes:
- maior massa de `s=1` em scores altos sugere sinal util;
- sobreposicao forte indica limitacao do baseline e necessidade de refinamento.

Este diagnostico usa `score_pu_clipped` apenas para leitura visual padronizada.
Ele nao substitui o ranking operacional nem as metricas numericas calculadas com `score_pu_raw`.


## 9) B2 - Analise de sensibilidade ao estimador base

B2 mantem o mesmo split, o mesmo preprocessamento, a mesma hipotese SCAR e o mesmo wrapper `pulearn.ElkanotoPuClassifier`.

A unica mudanca deliberada nesta secao e o estimador base nao linear:
- nesta trilha, B2 permanece operacionalmente configurado como `XGBoostClassifier`;
- um fallback para `HistGradientBoostingClassifier` continua documentado apenas como contingencia de ambiente, e nao como reabertura da escolha metodologica principal.

Comentario metodologico obrigatorio:
- B2 e analise de sensibilidade ao estimador base, nao nova hipotese PU;
- o objetivo e verificar o impacto da nao linearidade sem alterar o arcabouco SCAR.

In [ ]:
# ---------------------------------------------------------
# B2 - Analise de sensibilidade ao estimador base
# ---------------------------------------------------------
# B2 preserva o mesmo fluxo PU classico sob SCAR e troca apenas o
# estimador base interno do wrapper Elkan-Noto.

if XGBOOST_AVAILABLE:
    b2_base_estimator_name = "XGBoostClassifier"
    b2_estimator_reason = "XGBoost disponivel; B2 usa um unico estimador nao linear dentro do mesmo wrapper PU."
    estimator_b2 = XGBClassifier(
        n_estimators=160 if SMOKE else 320,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_weight=20,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=4,
        verbosity=0,
    )
else:
    b2_base_estimator_name = "HistGradientBoostingClassifier"
    b2_estimator_reason = "XGBoost indisponivel; B2 usa HistGradientBoostingClassifier como alternativa nao linear nativa."
    estimator_b2 = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=160 if SMOKE else 320,
        min_samples_leaf=200,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=SEED,
    )

print("Configuracao de B2:")
print(f"- estimador base: {b2_base_estimator_name}")
print(f"- justificativa: {b2_estimator_reason}")
if not XGBOOST_AVAILABLE and xgb_import_error is not None:
    print(f"- detalhe tecnico do fallback: {xgb_import_error}")

pu_b2 = pulearn.ElkanotoPuClassifier(estimator=estimator_b2, hold_out_ratio=HOLD_OUT_RATIO_MAIN, random_state=SEED)
model_b2 = Pipeline(steps=[("preprocess", preprocessor), ("pu", pu_b2)])

with warnings.catch_warnings(record=True) as w_list_b2:
    warnings.simplefilter("always")
    t0_b2 = time.perf_counter()
    model_b2.fit(X_train, y_train.to_numpy(dtype="int8", copy=False))
    train_seconds_b2 = time.perf_counter() - t0_b2

b2_warning_messages = [str(w.message) for w in w_list_b2]
b2_convergence_warnings = int(sum(1 for w in w_list_b2 if issubclass(w.category, ConvergenceWarning)))
b2_converged = bool(b2_convergence_warnings == 0)

pu_step_b2 = model_b2.named_steps["pu"]
score_val_b2, pos_idx_b2, class_source_b2, class_labels_b2 = predict_positive_score(model_b2, X_val)
score_test_b2, _, _, _ = predict_positive_score(model_b2, X_test)
auc_val_b2 = roc_auc_score(y_val, score_val_b2)
auc_test_b2, df_ranking_k_b2 = compute_ranking_metrics(y_test, score_test_b2)
c_estimado_b2 = getattr(pu_step_b2, "c", np.nan)

meta_cols_scores_b2 = [c for c in ["cpf_agricultor", "id_familia", "ibge", TARGET_COL] if c in df.columns]
df_scores_test_b2 = df.iloc[idx_test][meta_cols_scores_b2].copy().rename(columns={TARGET_COL: "s"})
df_scores_test_b2["indice_linha_original"] = df.iloc[idx_test].index.to_numpy()
df_scores_test_b2 = df_scores_test_b2.reset_index(drop=True)
df_scores_test_b2["score_pu_raw"] = score_test_b2.astype("float64")
df_scores_test_b2["score_pu_clipped"] = np.clip(score_test_b2, 0.0, 1.0).astype("float32")
df_scores_test_b2["codigo_modelo"] = "B2"

pct_score_raw_gt_1_b2 = float((df_scores_test_b2["score_pu_raw"] > 1).mean()) if len(df_scores_test_b2) > 0 else np.nan
ranking_unlabeled_b2 = (
    df_scores_test_b2[df_scores_test_b2["s"] == 0]
    .sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True])
    .reset_index(drop=True)
)

compare_b1_b2 = pd.DataFrame([
    {"codigo_modelo": "B1", "base_estimator": "LogisticRegression", "auc_proxy_s_val": float(auc_val_final), "auc_proxy_s_test": float(auc_s), "c_est": float(c_estimado) if pd.notna(c_estimado) else np.nan, "pct_score_raw_gt_1": float((df_scores_test["score_pu_raw"] > 1).mean()) if len(df_scores_test) > 0 else np.nan, "train_seconds": float(train_seconds_final), "converged": bool(converged_final)},
    {"codigo_modelo": "B2", "base_estimator": b2_base_estimator_name, "auc_proxy_s_val": float(auc_val_b2), "auc_proxy_s_test": float(auc_test_b2), "c_est": float(c_estimado_b2) if pd.notna(c_estimado_b2) else np.nan, "pct_score_raw_gt_1": pct_score_raw_gt_1_b2, "train_seconds": float(train_seconds_b2), "converged": bool(b2_converged)},
])

rank_compare_b1_b2 = (
    df_scores_test[["indice_linha_original", "cpf_agricultor", "s", "score_pu_raw"]]
    .rename(columns={"score_pu_raw": "score_b1_raw"})
    .merge(df_scores_test_b2[["indice_linha_original", "score_pu_raw"]].rename(columns={"score_pu_raw": "score_b2_raw"}), on="indice_linha_original", how="inner", validate="1:1")
)
rank_compare_b1_b2 = rank_compare_b1_b2[rank_compare_b1_b2["s"] == 0].copy()
df_topk_overlap_b1_b2 = pd.DataFrame([
    {"k_frac": frac, "topk_overlap": topk_overlap(rank_compare_b1_b2, "score_b1_raw", "score_b2_raw", frac)}
    for frac in (0.01, 0.05, 0.10)
])

print("\nComparacao resumida entre B1 e B2:")
display(compare_b1_b2)
print("\nMetricas de ranking de B2 no teste:")
display(df_ranking_k_b2)
print("\nSobreposicao do topo do ranking entre B1 e B2:")
display(df_topk_overlap_b1_b2)
if b2_warning_messages:
    print("\nAvisos capturados em B2 (ate 5):")
    for msg in b2_warning_messages[:5]:
        print(f"- {msg}")


### Interpretacao da Secao 9 (B2)

- B2 preserva a hipotese SCAR e troca apenas o estimador base interno do Elkan-Noto.
- Qualquer ganho de AUC proxy ou de Recall@k deve ser lido como sensibilidade ao estimador, nao como superacao da limitacao central do SCAR.
- O ranking oficial da trilha so muda se B2 superar B1 no criterio explicito de selecao do melhor modelo bruto.


## 10) B3 - Calibracao do melhor modelo entre B1 e B2

B3 escolhe entre B1 e B2 com base em `auc_proxy_s` na validacao, usando uma tolerancia pequena para evitar trocar o modelo bruto por diferencas irrelevantes na quarta casa decimal.

Depois dessa escolha, aplica-se uma calibracao monotona (`IsotonicRegression`) sobre os scores brutos de validacao.

Comentario metodologico obrigatorio:
- B3 e pos-processamento interpretativo, nao nova familia metodologica;
- a calibracao melhora a leitura do score, mas nao elimina a hipotese SCAR nem substitui o score bruto oficial para ranking operacional.


In [ ]:
# ---------------------------------------------------------
# B3 - Calibracao do melhor modelo bruto entre B1 e B2
# ---------------------------------------------------------
# O score calibrado e interpretativo. O score oficial do ranking continua
# sendo o score_pu_raw do melhor modelo bruto.

BEST_RAW_MODEL_TOL = 1e-4
auc_val_gap_b2_vs_b1 = float(auc_val_b2 - auc_val_final)

if auc_val_gap_b2_vs_b1 > BEST_RAW_MODEL_TOL:
    best_raw_model_code = "B2"
    best_raw_model_name = f"Elkan-Noto + {b2_base_estimator_name}"
    best_raw_base_estimator_name = b2_base_estimator_name
    best_raw_model = model_b2
    best_score_val_raw = np.asarray(score_val_b2, dtype="float64")
    best_score_test_raw = np.asarray(score_test_b2, dtype="float64")
    best_scores_test_df = df_scores_test_b2.copy()
    best_ranking_unlabeled = ranking_unlabeled_b2.copy()
    best_df_ranking_k = df_ranking_k_b2.copy()
    best_auc_val_raw = float(auc_val_b2)
    best_auc_test_raw = float(auc_test_b2)
    best_raw_c_est = float(c_estimado_b2) if pd.notna(c_estimado_b2) else np.nan
    best_raw_train_seconds = float(train_seconds_b2)
    best_raw_converged = bool(b2_converged)
    best_raw_class_source = str(class_source_b2)
    best_raw_class_labels = str(class_labels_b2)
    best_raw_pct_score_gt_1 = pct_score_raw_gt_1_b2
else:
    best_raw_model_code = "B1"
    best_raw_model_name = "Elkan-Noto + Logistic Regression"
    best_raw_base_estimator_name = "LogisticRegression"
    best_raw_model = model_final
    best_score_val_raw = np.asarray(score_val_final, dtype="float64")
    best_score_test_raw = np.asarray(score_test_final, dtype="float64")
    best_scores_test_df = df_scores_test.copy()
    best_ranking_unlabeled = ranking_unlabeled.copy()
    best_df_ranking_k = df_ranking_k.copy()
    best_auc_val_raw = float(auc_val_final)
    best_auc_test_raw = float(auc_s)
    best_raw_c_est = float(c_estimado) if pd.notna(c_estimado) else np.nan
    best_raw_train_seconds = float(train_seconds_final)
    best_raw_converged = bool(converged_final)
    best_raw_class_source = str(class_source_final)
    best_raw_class_labels = str(class_labels_final)
    best_raw_pct_score_gt_1 = float((df_scores_test["score_pu_raw"] > 1).mean()) if len(df_scores_test) > 0 else np.nan

print("Criterio de escolha do melhor modelo bruto para B3:")
print(f"- auc_val B1: {auc_val_final:.6f}")
print(f"- auc_val B2: {auc_val_b2:.6f}")
print(f"- delta B2 - B1 em validacao: {auc_val_gap_b2_vs_b1:.6f}")
print(f"- tolerancia: {BEST_RAW_MODEL_TOL:.0e}")
print(f"- modelo bruto escolhido: {best_raw_model_code} ({best_raw_model_name})")

y_val_np = pd.Series(y_val).astype("int8").to_numpy(copy=False)
y_test_np = pd.Series(y_test).astype("int8").to_numpy(copy=False)
auc_val_raw_selected, df_ranking_k_val_raw = compute_ranking_metrics(y_val_np, best_score_val_raw)

calibrator_b3 = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
t0_cal = time.perf_counter()
calibrator_b3.fit(best_score_val_raw, y_val_np)
calibration_seconds_b3 = time.perf_counter() - t0_cal

score_val_calibrated = calibrator_b3.predict(best_score_val_raw).astype("float64")
score_test_calibrated = calibrator_b3.predict(best_score_test_raw).astype("float64")
auc_val_b3, df_ranking_k_val_b3 = compute_ranking_metrics(y_val_np, score_val_calibrated)
auc_test_b3, df_ranking_k_b3 = compute_ranking_metrics(y_test_np, score_test_calibrated)

df_scores_test_b3 = best_scores_test_df[["cpf_agricultor", "id_familia", "ibge", "s", "indice_linha_original", "score_pu_raw", "score_pu_clipped"]].copy()
df_scores_test_b3 = df_scores_test_b3.rename(columns={"score_pu_raw": "score_pu_raw_origem", "score_pu_clipped": "score_pu_clipped_origem"})
df_scores_test_b3["score_pu_calibrated"] = score_test_calibrated.astype("float32")
df_scores_test_b3["codigo_modelo_origem"] = best_raw_model_code

ranking_unlabeled_b3 = (
    df_scores_test_b3[df_scores_test_b3["s"] == 0]
    .sort_values(["score_pu_calibrated", "cpf_agricultor"], ascending=[False, True])
    .reset_index(drop=True)
)

pct_score_calibrated_gt_1 = float((df_scores_test_b3["score_pu_calibrated"] > 1).mean()) if len(df_scores_test_b3) > 0 else np.nan
b3_before_after = pd.DataFrame([
    {"etapa": "validacao", "score": "raw_origem", "auc_proxy_s": float(auc_val_raw_selected)},
    {"etapa": "validacao", "score": "calibrated", "auc_proxy_s": float(auc_val_b3)},
    {"etapa": "teste", "score": "raw_origem", "auc_proxy_s": float(best_auc_test_raw)},
    {"etapa": "teste", "score": "calibrated", "auc_proxy_s": float(auc_test_b3)},
])

df_overlap_b3 = pd.DataFrame([
    {"k_frac": frac, "topk_overlap": topk_overlap(df_scores_test_b3[df_scores_test_b3["s"] == 0].copy(), "score_pu_raw_origem", "score_pu_calibrated", frac)}
    for frac in (0.01, 0.05, 0.10)
])

df_distribution_b3 = pd.DataFrame([
    {"serie": "raw_origem", "mean": float(df_scores_test_b3["score_pu_raw_origem"].mean()), "p50": float(df_scores_test_b3["score_pu_raw_origem"].quantile(0.50)), "p90": float(df_scores_test_b3["score_pu_raw_origem"].quantile(0.90)), "max": float(df_scores_test_b3["score_pu_raw_origem"].max())},
    {"serie": "clipped_origem", "mean": float(df_scores_test_b3["score_pu_clipped_origem"].mean()), "p50": float(df_scores_test_b3["score_pu_clipped_origem"].quantile(0.50)), "p90": float(df_scores_test_b3["score_pu_clipped_origem"].quantile(0.90)), "max": float(df_scores_test_b3["score_pu_clipped_origem"].max())},
    {"serie": "calibrated", "mean": float(df_scores_test_b3["score_pu_calibrated"].mean()), "p50": float(df_scores_test_b3["score_pu_calibrated"].quantile(0.50)), "p90": float(df_scores_test_b3["score_pu_calibrated"].quantile(0.90)), "max": float(df_scores_test_b3["score_pu_calibrated"].max())},
])

print("\nComparacao antes/depois da calibracao:")
display(b3_before_after)
print("\nMudanca na distribuicao do score:")
display(df_distribution_b3)
print("\nSobreposicao do topo do ranking entre bruto e calibrado:")
display(df_overlap_b3)
print(f"\nTempo de calibracao: {calibration_seconds_b3:.2f}s")
print(f"Scores calibrados > 1: {pct_score_calibrated_gt_1:.4%}")


### Interpretacao da Secao 10 (B3)

- B3 entrega uma versao calibrada do melhor score bruto entre B1 e B2 e mostra o efeito da calibracao sobre AUC proxy, distribuicao do score e topo do ranking.
- O score calibrado existe para apoiar interpretacao e tabelas em escala `[0,1]`.
- O score oficial da trilha para ranking operacional continua sendo `score_pu_raw` do melhor modelo bruto.


## 11) Diagnostico especifico dos casos com `score_pu_raw > 1`

Esta secao aprofunda um fenomeno tipico do PU classico por reponderacao probabilistica: a ocorrencia de scores brutos acima de `1` apos a correcao por `c`.

Objetivos desta leitura:
- transformar um aviso tecnico em diagnostico interpretavel;
- comparar o perfil dos casos com `score_pu_raw > 1` com o restante da base de teste;
- observar distribuicao territorial, posicao no ranking e efeito do clipping/calibracao;
- reforcar por que `score_pu_raw` e score de ranking, nao probabilidade calibrada literal.

In [ ]:
# ---------------------------------------------------------
# Diagnostico aprofundado dos casos com score_pu_raw > 1
# ---------------------------------------------------------
# Contexto:
# Esta leitura usa o melhor modelo bruto da Trilha B, isto e,
# o mesmo score oficial que sustenta o ranking operacional.
#
# Objetivo estatistico:
# Comparar os casos com score_pu_raw > 1 ao restante da base de
# teste para entender se esse comportamento se concentra em perfis,
# territorios ou posicoes especificas do ranking.
#
# Interpretacao:
# Valores acima de 1 nao representam "erro". Eles decorrem da
# correcao por c no metodo de Elkan & Noto e reforcam o papel do
# score bruto como indice de ordenacao, nao como probabilidade
# literalmente calibrada.

region_map = {1: "Norte", 2: "Nordeste", 3: "Sudeste", 4: "Sul", 5: "Centro-Oeste"}
uf_map = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ", 35: "SP", 41: "PR",
    42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF",
}
context_candidate_cols = [
    "id_regiao", "cd_uf", "idade_2025", "jovem", "id_sexo",
    "st_assentado_sipra", "st_trabalha_ufpr", "vl_renda_total_auferida", "vl_area_total_ha",
]
context_cols_available = [c for c in context_candidate_cols if c in df.columns]

df_test_context = (
    df.iloc[idx_test][["cpf_agricultor", "id_familia", "ibge", TARGET_COL] + context_cols_available]
    .copy()
    .rename(columns={TARGET_COL: "s"})
)
df_test_context["indice_linha_original"] = df.iloc[idx_test].index.to_numpy()
df_test_context = df_test_context.reset_index(drop=True)

df_score_gt1_base = (
    best_scores_test_df[["cpf_agricultor", "id_familia", "ibge", "s", "indice_linha_original", "score_pu_raw", "score_pu_clipped"]]
    .copy()
    .merge(df_scores_test_b3[["indice_linha_original", "score_pu_calibrated"]], on="indice_linha_original", how="left", validate="1:1")
    .merge(df_test_context.drop(columns=["cpf_agricultor", "id_familia", "ibge", "s"]), on="indice_linha_original", how="left", validate="1:1")
)

df_score_gt1_base["score_gt_1_flag"] = df_score_gt1_base["score_pu_raw"] > 1
df_score_gt1_base["grupo_score_raw"] = np.where(df_score_gt1_base["score_gt_1_flag"], "score_pu_raw_gt_1", "score_pu_raw_le_1")
if "id_regiao" in df_score_gt1_base.columns:
    df_score_gt1_base["regiao_nome"] = pd.to_numeric(df_score_gt1_base["id_regiao"], errors="coerce").astype("Int64").map(region_map)
else:
    df_score_gt1_base["regiao_nome"] = pd.NA
if "cd_uf" in df_score_gt1_base.columns:
    df_score_gt1_base["uf_sigla"] = pd.to_numeric(df_score_gt1_base["cd_uf"], errors="coerce").astype("Int64").map(uf_map)
else:
    df_score_gt1_base["uf_sigla"] = pd.NA

df_score_gt1_base = df_score_gt1_base.sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True]).reset_index(drop=True)
df_score_gt1_base["ranking_global_raw"] = np.arange(1, len(df_score_gt1_base) + 1, dtype="int64")
df_unlabeled_rank_pos = (
    df_score_gt1_base[df_score_gt1_base["s"] == 0]
    .sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True])
    .reset_index(drop=True)[["indice_linha_original"]]
    .copy()
)
df_unlabeled_rank_pos["ranking_unlabeled_raw"] = np.arange(1, len(df_unlabeled_rank_pos) + 1, dtype="int64")
df_score_gt1_base = df_score_gt1_base.merge(df_unlabeled_rank_pos, on="indice_linha_original", how="left", validate="1:1")

profile_rows = []
for group_name, df_group in df_score_gt1_base.groupby("grupo_score_raw"):
    row = {
        "grupo_score_raw": group_name,
        "n_casos": int(len(df_group)),
        "pct_casos_teste": float(len(df_group) / len(df_score_gt1_base)) if len(df_score_gt1_base) > 0 else np.nan,
        "pct_s_1": float(df_group["s"].mean()) if len(df_group) > 0 else np.nan,
        "pct_unlabeled": float((df_group["s"] == 0).mean()) if len(df_group) > 0 else np.nan,
        "mediana_score_raw": float(df_group["score_pu_raw"].median()) if len(df_group) > 0 else np.nan,
        "p90_score_raw": float(df_group["score_pu_raw"].quantile(0.90)) if len(df_group) > 0 else np.nan,
        "mediana_score_clipped": float(df_group["score_pu_clipped"].median()) if len(df_group) > 0 else np.nan,
        "mediana_score_calibrated": float(df_group["score_pu_calibrated"].median()) if len(df_group) > 0 else np.nan,
        "mediana_rank_global": float(df_group["ranking_global_raw"].median()) if len(df_group) > 0 else np.nan,
        "mediana_rank_unlabeled": float(df_group["ranking_unlabeled_raw"].dropna().median()) if df_group["ranking_unlabeled_raw"].notna().any() else np.nan,
    }
    if "idade_2025" in df_group.columns:
        row["idade_mediana"] = float(pd.to_numeric(df_group["idade_2025"], errors="coerce").median())
    if "jovem" in df_group.columns:
        row["pct_jovem"] = float(pd.to_numeric(df_group["jovem"], errors="coerce").mean())
    if "st_assentado_sipra" in df_group.columns:
        row["pct_assentado"] = float(pd.to_numeric(df_group["st_assentado_sipra"], errors="coerce").mean())
    if "st_trabalha_ufpr" in df_group.columns:
        row["pct_trabalha_ufpr"] = float(pd.to_numeric(df_group["st_trabalha_ufpr"], errors="coerce").mean())
    if "vl_renda_total_auferida" in df_group.columns:
        row["renda_total_mediana"] = float(pd.to_numeric(df_group["vl_renda_total_auferida"], errors="coerce").median())
    if "vl_area_total_ha" in df_group.columns:
        row["area_total_mediana_ha"] = float(pd.to_numeric(df_group["vl_area_total_ha"], errors="coerce").median())
    profile_rows.append(row)

df_score_raw_gt1_profile = pd.DataFrame(profile_rows)
if df_score_gt1_base["regiao_nome"].notna().any():
    df_score_raw_gt1_regiao = df_score_gt1_base.groupby(["grupo_score_raw", "regiao_nome"], dropna=False).size().rename("n").reset_index()
    df_score_raw_gt1_regiao["pct_no_grupo"] = df_score_raw_gt1_regiao.groupby("grupo_score_raw")["n"].transform(lambda s: s / s.sum())
    df_score_raw_gt1_regiao = df_score_raw_gt1_regiao.sort_values(["grupo_score_raw", "pct_no_grupo"], ascending=[True, False]).reset_index(drop=True)
else:
    df_score_raw_gt1_regiao = pd.DataFrame(columns=["grupo_score_raw", "regiao_nome", "n", "pct_no_grupo"])
if df_score_gt1_base["uf_sigla"].notna().any():
    df_score_raw_gt1_uf = df_score_gt1_base.groupby(["grupo_score_raw", "uf_sigla"], dropna=False).size().rename("n").reset_index()
    df_score_raw_gt1_uf["pct_no_grupo"] = df_score_raw_gt1_uf.groupby("grupo_score_raw")["n"].transform(lambda s: s / s.sum())
    df_score_raw_gt1_uf = df_score_raw_gt1_uf.sort_values(["grupo_score_raw", "pct_no_grupo"], ascending=[True, False]).reset_index(drop=True)
else:
    df_score_raw_gt1_uf = pd.DataFrame(columns=["grupo_score_raw", "uf_sigla", "n", "pct_no_grupo"])

df_score_raw_gt1_examples = (
    df_score_gt1_base[df_score_gt1_base["score_gt_1_flag"]]
    .sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True])
    .head(15)
    .copy()
)

diag_tables = []
if not df_score_raw_gt1_profile.empty:
    diag_tables.append(
        df_score_raw_gt1_profile.melt(id_vars=["grupo_score_raw", "n_casos"], var_name="metrica", value_name="valor")
        .assign(secao="perfil", categoria=pd.NA)
        .rename(columns={"grupo_score_raw": "grupo"})[["secao", "grupo", "categoria", "n_casos", "metrica", "valor"]]
    )
if not df_score_raw_gt1_regiao.empty:
    diag_tables.append(
        df_score_raw_gt1_regiao.assign(secao="regiao", categoria=df_score_raw_gt1_regiao["regiao_nome"].astype("string"), metrica="pct_no_grupo", valor=df_score_raw_gt1_regiao["pct_no_grupo"].astype("float64"))
        .rename(columns={"grupo_score_raw": "grupo"})[["secao", "grupo", "categoria", "n", "metrica", "valor"]]
        .rename(columns={"n": "n_casos"})
    )
if not df_score_raw_gt1_uf.empty:
    diag_tables.append(
        df_score_raw_gt1_uf.assign(secao="uf", categoria=df_score_raw_gt1_uf["uf_sigla"].astype("string"), metrica="pct_no_grupo", valor=df_score_raw_gt1_uf["pct_no_grupo"].astype("float64"))
        .rename(columns={"grupo_score_raw": "grupo"})[["secao", "grupo", "categoria", "n", "metrica", "valor"]]
        .rename(columns={"n": "n_casos"})
    )
if diag_tables:
    df_trilhaB_score_raw_gt1_diag = pd.concat(diag_tables, ignore_index=True)
else:
    df_trilhaB_score_raw_gt1_diag = pd.DataFrame(columns=["secao", "grupo", "categoria", "n_casos", "metrica", "valor"])

print("Resumo comparativo dos casos com score_pu_raw > 1:")
display(df_score_raw_gt1_profile)
if not df_score_raw_gt1_regiao.empty:
    print("\nDistribuicao regional por grupo de score bruto:")
    display(df_score_raw_gt1_regiao)
if not df_score_raw_gt1_uf.empty:
    print("\nTop 10 UFs por grupo de score bruto:")
    display(df_score_raw_gt1_uf.groupby("grupo_score_raw", group_keys=False).head(10))
if df_score_raw_gt1_examples.empty:
    print("\nNao ha casos com score_pu_raw > 1 no melhor modelo bruto.")
else:
    example_cols = [c for c in ["cpf_agricultor", "id_familia", "ibge", "regiao_nome", "uf_sigla", "s", "score_pu_raw", "score_pu_clipped", "score_pu_calibrated", "ranking_global_raw", "ranking_unlabeled_raw"] if c in df_score_raw_gt1_examples.columns]
    print("\nExemplos de casos com score_pu_raw > 1 (CPF mascarado):")
    display(mask_cpf_frame(df_score_raw_gt1_examples[example_cols], columns=["cpf_agricultor"]))

### Interpretacao da Secao 11 (score bruto acima de 1)

- `score_pu_raw > 1` e um comportamento esperado sob a correcao por `c`, nao um bug numerico isolado.
- O diagnostico compara esses casos com o restante da base para verificar se eles se concentram em perfis ou territorios especificos.
- `score_pu_clipped` e `score_pu_calibrated` ajudam a ler a escala, mas nao substituem o papel do `score_pu_raw` como score oficial de ranking.

## 12) Estabilidade do ranking entre B2 e B3

Esta secao mede diretamente o quanto a calibracao monotona de B3 altera ou preserva o ranking operacional produzido por B2.

O objetivo e responder a uma pergunta simples:
- B3 apenas reescala o score para leitura interpretativa?
- ou muda materialmente quem sobe para o topo da fila analitica?

In [ ]:
# ---------------------------------------------------------
# Estabilidade do ranking entre B2 bruto e B3 calibrado
# ---------------------------------------------------------
# Contexto:
# B3 e um pos-processamento monotono sobre o melhor modelo bruto.
#
# Objetivo estatistico:
# Medir se a calibracao altera a identidade dos casos priorizados no
# topo do ranking operacional, para alem da simples mudanca de escala.
#
# Interpretacao:
# Overlap elevado e correlacoes proximas de 1 sustentam a leitura
# de que B3 e interpretativo; quedas relevantes sugeririam que a
# calibracao muda a priorizacao de forma substantiva.

df_b2_b3_compare = (
    df_scores_test_b3[df_scores_test_b3["s"] == 0]
    .copy()
    .sort_values(["score_pu_raw_origem", "cpf_agricultor"], ascending=[False, True])
    .reset_index(drop=True)
)

overlap_rows = []
for frac in TOP_RANK_PCTS:
    overlap_rows.append({
        "tipo_recorte": "fracao",
        "recorte": f"top_{int(round(frac * 100))}pct",
        "k_absoluto": max(1, int(np.ceil(len(df_b2_b3_compare) * frac))),
        "overlap": topk_overlap(df_b2_b3_compare, "score_pu_raw_origem", "score_pu_calibrated", frac),
    })
for top_n in TOP_RANK_ABS:
    overlap_rows.append({
        "tipo_recorte": "absoluto",
        "recorte": f"top_{top_n}",
        "k_absoluto": int(min(top_n, len(df_b2_b3_compare))),
        "overlap": topn_overlap(df_b2_b3_compare, "score_pu_raw_origem", "score_pu_calibrated", top_n),
    })

df_overlap_b3_detalhado = pd.DataFrame(overlap_rows)
correlation_pairs = [
    ("score_pu_raw_origem", "score_pu_clipped_origem"),
    ("score_pu_raw_origem", "score_pu_calibrated"),
    ("score_pu_clipped_origem", "score_pu_calibrated"),
]
df_corr_b3 = pd.DataFrame([
    {
        "serie_a": left_col,
        "serie_b": right_col,
        "pearson": safe_corr(df_b2_b3_compare[left_col], df_b2_b3_compare[right_col], method="pearson"),
        "spearman": safe_corr(df_b2_b3_compare[left_col], df_b2_b3_compare[right_col], method="spearman"),
    }
    for left_col, right_col in correlation_pairs
])

overlap_export = df_overlap_b3_detalhado.assign(secao="overlap", serie_a=pd.NA, serie_b=pd.NA, pearson=np.nan, spearman=np.nan)[["secao", "tipo_recorte", "recorte", "k_absoluto", "overlap", "serie_a", "serie_b", "pearson", "spearman"]]
corr_export = df_corr_b3.assign(secao="correlacao", tipo_recorte=pd.NA, recorte=pd.NA, k_absoluto=np.nan, overlap=np.nan)[["secao", "tipo_recorte", "recorte", "k_absoluto", "overlap", "serie_a", "serie_b", "pearson", "spearman"]]
df_trilhaB_b2_b3_ranking_overlap = pd.concat([overlap_export, corr_export], ignore_index=True)

print("Sobreposicao do topo do ranking entre B2 bruto e B3 calibrado:")
display(df_overlap_b3_detalhado)

print("\nCorrelacoes entre score bruto, clipped e calibrado:")
display(df_corr_b3)

### Interpretacao da Secao 12 (estabilidade B2 vs B3)

- Overlaps altos no topo do ranking reforcam que B3 atua principalmente como reescala interpretativa.
- Correlacoes muito elevadas entre `score_pu_raw`, `score_pu_clipped` e `score_pu_calibrated` indicam preservacao da hierarquia relativa dos casos.
- Se algum recorte mostrar queda importante de overlap, essa perda precisa ser lida como custo de interpretabilidade sobre a priorizacao operacional.

## 13) Sensibilidade estrutural a `hold_out_ratio` e `c_est`

Nesta secao, variamos apenas `hold_out_ratio` mantendo congelados o split, o preprocessamento e o estimador nao linear de B2.

Essa analise funciona como o principal teste de robustez estrutural da Trilha B, pois `hold_out_ratio` interfere diretamente na estimativa de `c` e, portanto, na escala do score sob SCAR.

In [ ]:
# ---------------------------------------------------------
# Sensibilidade do PU classico a hold_out_ratio
# ---------------------------------------------------------
# Contexto:
# O wrapper Elkan-Noto usa hold_out_ratio para reservar parte dos
# positivos rotulados e estimar a constante c.
#
# Objetivo estatistico:
# Verificar se pequenas mudancas nessa escolha estrutural afetam
# AUC_s, Recall@k, Precision@k, c estimado e a identidade do topo
# do ranking quando todo o restante do pipeline e mantido fixo.
#
# Interpretacao:
# Se as metricas e o topo do ranking permanecerem estaveis, a
# leitura sob SCAR fica mais robusta. Se a variacao for grande, o
# metodo depende materialmente da escolha de hold_out_ratio.

holdout_rows = []
reference_unlabeled_b2 = ranking_unlabeled_b2[["cpf_agricultor", "score_pu_raw"]].rename(columns={"score_pu_raw": "score_ref_main"})

for hold_out_ratio in HOLD_OUT_RATIO_GRID:
    estimator_holdout = clone(estimator_b2)
    preprocessor_holdout = clone(preprocessor)
    pu_holdout = pulearn.ElkanotoPuClassifier(estimator=estimator_holdout, hold_out_ratio=hold_out_ratio, random_state=SEED)
    model_holdout = Pipeline(steps=[("preprocess", preprocessor_holdout), ("pu", pu_holdout)])

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        t0_holdout = time.perf_counter()
        model_holdout.fit(X_train, y_train.to_numpy(dtype="int8", copy=False))
        train_seconds_holdout = time.perf_counter() - t0_holdout

    pu_step_holdout = model_holdout.named_steps["pu"]
    score_val_holdout, _, _, _ = predict_positive_score(model_holdout, X_val)
    score_test_holdout, _, _, _ = predict_positive_score(model_holdout, X_test)
    auc_val_holdout = float(roc_auc_score(y_val, score_val_holdout))
    auc_test_holdout, df_rank_holdout = compute_ranking_metrics(y_test, score_test_holdout)
    c_attr_holdout = getattr(pu_step_holdout, "c", np.nan)
    c_est_holdout = float(c_attr_holdout) if pd.notna(c_attr_holdout) else np.nan

    df_scores_holdout = (
        df.iloc[idx_test][["cpf_agricultor", "id_familia", "ibge", TARGET_COL]]
        .copy()
        .rename(columns={TARGET_COL: "s"})
        .reset_index(drop=True)
    )
    df_scores_holdout["score_pu_raw"] = score_test_holdout.astype("float64")

    ranking_unlabeled_holdout = (
        df_scores_holdout[df_scores_holdout["s"] == 0]
        .sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True])
        .reset_index(drop=True)
    )

    df_compare_holdout = reference_unlabeled_b2.merge(
        ranking_unlabeled_holdout[["cpf_agricultor", "score_pu_raw"]].rename(columns={"score_pu_raw": "score_candidate"}),
        on="cpf_agricultor",
        how="inner",
        validate="1:1",
    )

    holdout_rows.append(
        {
            "hold_out_ratio": float(hold_out_ratio),
            "auc_proxy_s_val": float(auc_val_holdout),
            "auc_proxy_s": float(auc_test_holdout),
            "recall_at_1pct": metric_at(df_rank_holdout, 0.01, "recall_at_k"),
            "recall_at_5pct": metric_at(df_rank_holdout, 0.05, "recall_at_k"),
            "recall_at_10pct": metric_at(df_rank_holdout, 0.10, "recall_at_k"),
            "precision_at_1pct": metric_at(df_rank_holdout, 0.01, "precision_at_k"),
            "precision_at_5pct": metric_at(df_rank_holdout, 0.05, "precision_at_k"),
            "precision_at_10pct": metric_at(df_rank_holdout, 0.10, "precision_at_k"),
            "c_est": c_est_holdout,
            "pct_score_raw_gt_1": float((df_scores_holdout["score_pu_raw"] > 1).mean()) if len(df_scores_holdout) > 0 else np.nan,
            "train_seconds": float(train_seconds_holdout),
            "convergence_warnings": int(len(captured_warnings)),
            "overlap_top_1pct_vs_main": topk_overlap(df_compare_holdout, "score_ref_main", "score_candidate", 0.01),
            "overlap_top_5pct_vs_main": topk_overlap(df_compare_holdout, "score_ref_main", "score_candidate", 0.05),
            "overlap_top_10pct_vs_main": topk_overlap(df_compare_holdout, "score_ref_main", "score_candidate", 0.10),
            "overlap_top_1000_vs_main": topn_overlap(df_compare_holdout, "score_ref_main", "score_candidate", 1000),
            "overlap_top_5000_vs_main": topn_overlap(df_compare_holdout, "score_ref_main", "score_candidate", 5000),
        }
    )

df_holdout_sensitivity = pd.DataFrame(holdout_rows).sort_values("hold_out_ratio").reset_index(drop=True)

print("Sensibilidade estrutural a hold_out_ratio:")
display(df_holdout_sensitivity)

### Interpretacao da Secao 13 (sensibilidade a `hold_out_ratio`)

- `hold_out_ratio` afeta o mecanismo de correcao do PU classico porque entra na estimativa de `c`.
- Estabilidade de `AUC_s`, `Recall@k` e do topo do ranking sugere robustez do mecanismo de correcao SCAR.
- Se `c_est` e `% score_pu_raw > 1` variam muito com a grade testada, a leitura do score fica mais sensivel a essa escolha estrutural.

## 14) Interpretabilidade nao linear de B2 com SHAP

Esta secao introduz a principal lacuna que faltava na Trilha B: interpretabilidade do modelo nao linear de sensibilidade (`B2`).

Decisao metodologica adotada:
- o SHAP incide sobre o componente de score aprendido pelo estimador base nao linear do B2;
- a correcao por `c` continua sendo parte da escala PU final, mas nao e o objeto diretamente explicado pelo SHAP;
- a leitura segue condicionada a SCAR e nao deve ser interpretada como causalidade em `Y`.

In [ ]:
# ---------------------------------------------------------
# SHAP global do componente nao linear de B2
# ---------------------------------------------------------
# Contexto:
# O objetivo aqui nao e "explicar y latente", mas abrir a caixa
# preta do estimador base nao linear que sustenta o ranking de B2.
#
# Objetivo estatistico:
# Medir a contribuicao media das features transformadas para o score
# aprendido pelo estimador base do Elkan-Noto.
#
# Interpretacao:
# Features com maior mean(|SHAP|) aparecem como os principais
# condutores da estrutura preditiva nao linear de B2 sob SCAR.

shap_feature_names_b2 = []
shap_engine_b2 = None
shap_status_b2 = "nao_executado"
shap_reason_b2 = None

df_trilhaB_shap_global = pd.DataFrame(columns=["feature", "mean_abs_shap", "mean_shap", "segmento", "n_linhas_segmento", "n_linhas_usadas_shap", "amostrado", "tempo_segundos"])
df_trilhaB_shap_top = pd.DataFrame(columns=["feature", "mean_abs_shap", "mean_shap", "segmento", "n_linhas_segmento", "n_linhas_usadas_shap", "amostrado", "tempo_segundos"])
df_trilhaB_shap_groups = pd.DataFrame(columns=["grupo", "coluna_grupo", "valor_grupo", "n_linhas_grupo", "feature", "mean_abs_shap", "mean_shap"])
df_trilhaB_shap_sintese = pd.DataFrame(columns=["feature", "aparicoes", "fontes_distintas", "mean_abs_shap_media"])

if not XGBOOST_AVAILABLE or xgb is None:
    shap_status_b2 = "indisponivel"
    shap_reason_b2 = "xgboost nao esta disponivel no ambiente."
elif b2_base_estimator_name != "XGBoostClassifier":
    shap_status_b2 = "indisponivel"
    shap_reason_b2 = "B2 nao foi treinado com XGBoostClassifier nesta execucao; o bloco SHAP foi preservado, mas sem execucao automatica."
else:
    preprocess_b2 = model_b2.named_steps["preprocess"]
    X_test_b2_transformado = preprocess_b2.transform(X_test)
    if hasattr(X_test_b2_transformado, "toarray"):
        X_test_b2_transformado = X_test_b2_transformado.toarray()
    X_test_b2_transformado = np.asarray(X_test_b2_transformado, dtype="float32")

    shap_feature_names_b2 = [
        str(col) for col in (
            preprocess_b2.get_feature_names_out()
            if hasattr(preprocess_b2, "get_feature_names_out")
            else numeric_feature_cols
        )
    ]

    context_cols_prefixed = {col: f"ctx__{col}" for col in context_cols_available}
    context_cols_shap = ["cpf_agricultor", "id_familia", "ibge", "s", "indice_linha_original"] + list(context_cols_prefixed.values())
    df_context_shap_b2 = (
        df.iloc[idx_test][["cpf_agricultor", "id_familia", "ibge", TARGET_COL] + context_cols_available]
        .copy()
        .rename(columns={TARGET_COL: "s"})
        .reset_index(drop=True)
    )
    df_context_shap_b2 = df_context_shap_b2.rename(columns=context_cols_prefixed)
    df_context_shap_b2["indice_linha_original"] = df.iloc[idx_test].index.to_numpy()

    df_b2_X_test_transformado = pd.DataFrame(X_test_b2_transformado, columns=shap_feature_names_b2)
    df_b2_shap_base = pd.concat([df_context_shap_b2[context_cols_shap].reset_index(drop=True), df_b2_X_test_transformado.reset_index(drop=True)], axis=1)
    df_b2_shap_base = df_b2_shap_base.merge(df_scores_test_b2[["indice_linha_original", "score_pu_raw", "score_pu_clipped"]], on="indice_linha_original", how="inner", validate="1:1")

    pu_step_shap_b2 = model_b2.named_steps["pu"]
    b2_inner_estimator = getattr(pu_step_shap_b2, "estimator_", getattr(pu_step_shap_b2, "estimator", None))
    if b2_inner_estimator is None or not hasattr(b2_inner_estimator, "get_booster"):
        raise AttributeError("Nao foi possivel acessar um estimador XGBoost ajustado dentro do wrapper Elkan-Noto.")
    booster_b2 = b2_inner_estimator.get_booster()
    shap_engine_b2 = "xgboost_pred_contribs"
    df_b2_global_used, shap_global_values_b2, base_global_values_b2, df_trilhaB_shap_global = compute_segment_shap_xgb(
        df_segment=df_b2_shap_base,
        segment_name="global_teste_b2",
        booster=booster_b2,
        feature_names=shap_feature_names_b2,
        max_rows=MAX_SHAP_ROWS_GLOBAL,
        seed=SEED,
    )
    shap_status_b2 = "executado"

    print(f"Engine SHAP usada em B2: {shap_engine_b2}")
    print("Top 20 features no SHAP global de B2:")
    display(df_trilhaB_shap_global.head(20))

    plot_global = df_trilhaB_shap_global.head(PLOT_MAX_DISPLAY).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(plot_global["feature"], plot_global["mean_abs_shap"])
    plt.title("Trilha B - Importancia Global de B2 (mean|SHAP|)")
    plt.xlabel("mean(|SHAP|)")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    if SHAP_AVAILABLE:
        exp_global_b2 = shap.Explanation(
            values=shap_global_values_b2,
            base_values=base_global_values_b2,
            data=df_b2_global_used[shap_feature_names_b2].to_numpy(),
            feature_names=shap_feature_names_b2,
        )
        plt.figure(figsize=(11, 7))
        shap.plots.beeswarm(exp_global_b2, max_display=PLOT_MAX_DISPLAY, show=False)
        plt.title("Trilha B - SHAP Beeswarm Global de B2")
        plt.tight_layout()
        plt.show()
    else:
        print("Pacote shap indisponivel; beeswarm foi pulado, mas as tabelas de importancia foram geradas.")

if shap_status_b2 != "executado":
    print("Bloco SHAP de B2 nao executado.")
    if shap_reason_b2 is not None:
        print(f"Motivo: {shap_reason_b2}")

In [ ]:
# ---------------------------------------------------------
# SHAP no topo do ranking e por grupos analiticos
# ---------------------------------------------------------
# Contexto:
# A leitura global mostra o que move o score de B2 no teste como
# um todo. Agora observamos os segmentos mais priorizados.
#
# Objetivo estatistico:
# Comparar a importancia das features no topo do ranking unlabeled
# e em grupos selecionados, sem misturar isso com causalidade.
#
# Interpretacao:
# Se as features do topo diferirem do global, isso sugere que a
# logica do ranking nos percentis superiores e mais especifica do
# que a media do conjunto de teste.

if shap_status_b2 == "executado":
    df_unlabeled_shap_b2 = (
        df_b2_shap_base[df_b2_shap_base["s"] == 0]
        .sort_values(["score_pu_raw", "cpf_agricultor"], ascending=[False, True])
        .reset_index(drop=True)
    )

    segment_tables = []
    segment_cache = {}
    for pct in TOP_RANK_PCTS:
        n_top = max(1, int(np.ceil(len(df_unlabeled_shap_b2) * pct)))
        segment_name = f"top_{int(round(pct * 100))}pct_unlabeled"
        df_segment = df_unlabeled_shap_b2.head(n_top).copy()
        df_used, shap_values, base_values, df_imp = compute_segment_shap_xgb(
            df_segment=df_segment,
            segment_name=segment_name,
            booster=booster_b2,
            feature_names=shap_feature_names_b2,
            max_rows=MAX_SHAP_ROWS_TOP,
            seed=SEED,
        )
        segment_tables.append(df_imp)
        segment_cache[segment_name] = {"df_used": df_used.reset_index(drop=True), "shap_values": shap_values, "base_values": base_values}

    df_trilhaB_shap_top = pd.concat(segment_tables, ignore_index=True)
    print("Top features por segmento do ranking de B2:")
    display(df_trilhaB_shap_top.groupby("segmento", group_keys=False).head(10)[["segmento", "feature", "mean_abs_shap", "mean_shap", "n_linhas_usadas_shap"]])

    for segment_name in sorted(df_trilhaB_shap_top["segmento"].unique()):
        df_plot = df_trilhaB_shap_top[df_trilhaB_shap_top["segmento"] == segment_name].head(12).iloc[::-1]
        plt.figure(figsize=(9, 6))
        plt.barh(df_plot["feature"], df_plot["mean_abs_shap"])
        plt.title(f"Trilha B - {segment_name} (mean|SHAP|)")
        plt.xlabel("mean(|SHAP|)")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.show()

    ref_segment_name = "top_10pct_unlabeled"
    group_candidates = {
        "jovens_vs_nao_jovens": ["ctx__jovem"],
        "assentados_vs_nao_assentados": ["ctx__st_assentado_sipra"],
        "sexo": ["ctx__id_sexo"],
        "regiao": ["ctx__id_regiao"],
    }
    if ref_segment_name in segment_cache:
        df_ref_groups = segment_cache[ref_segment_name]["df_used"].reset_index(drop=True)
        shap_ref_groups = segment_cache[ref_segment_name]["shap_values"]
        group_tables = []
        for group_name, candidates in group_candidates.items():
            chosen = next((c for c in candidates if c in df_ref_groups.columns), None)
            if chosen is None:
                continue
            value_counts = df_ref_groups[chosen].value_counts(dropna=False)
            for raw_value, n_rows in value_counts.items():
                if int(n_rows) < MIN_GROUP_ROWS:
                    continue
                mask = (df_ref_groups[chosen] == raw_value).to_numpy()
                if int(mask.sum()) == 0:
                    continue
                shap_subset = shap_ref_groups[mask]
                df_group_imp = pd.DataFrame({
                    "grupo": group_name,
                    "coluna_grupo": chosen,
                    "valor_grupo": str(raw_value),
                    "n_linhas_grupo": int(mask.sum()),
                    "feature": shap_feature_names_b2,
                    "mean_abs_shap": np.abs(shap_subset).mean(axis=0),
                    "mean_shap": shap_subset.mean(axis=0),
                }).sort_values("mean_abs_shap", ascending=False, ignore_index=True)
                group_tables.append(df_group_imp)
        if group_tables:
            df_trilhaB_shap_groups = pd.concat(group_tables, ignore_index=True)
            print("Top features por grupo analitico em B2:")
            display(df_trilhaB_shap_groups.groupby(["grupo", "valor_grupo"], group_keys=False).head(10)[["grupo", "valor_grupo", "n_linhas_grupo", "feature", "mean_abs_shap", "mean_shap"]])

    df_global_top10_b2 = df_trilhaB_shap_global.head(10).copy(); df_global_top10_b2["fonte"] = "global"
    df_top_segments_top10_b2 = df_trilhaB_shap_top.groupby("segmento", group_keys=False).head(10).copy(); df_top_segments_top10_b2["fonte"] = "top_ranking"
    if df_trilhaB_shap_groups.empty:
        df_groups_top5_b2 = pd.DataFrame(columns=["feature", "mean_abs_shap", "fonte"])
    else:
        df_groups_top5_b2 = df_trilhaB_shap_groups.groupby(["grupo", "valor_grupo"], group_keys=False).head(5).copy(); df_groups_top5_b2["fonte"] = "grupos"

    df_recurrence_b2 = pd.concat([
        df_global_top10_b2[["feature", "mean_abs_shap", "fonte"]],
        df_top_segments_top10_b2[["feature", "mean_abs_shap", "fonte"]],
        df_groups_top5_b2[["feature", "mean_abs_shap", "fonte"]],
    ], ignore_index=True)

    df_trilhaB_shap_sintese = (
        df_recurrence_b2.groupby("feature", as_index=False)
        .agg(aparicoes=("feature", "size"), fontes_distintas=("fonte", "nunique"), mean_abs_shap_media=("mean_abs_shap", "mean"))
        .sort_values(["aparicoes", "fontes_distintas", "mean_abs_shap_media"], ascending=[False, False, False], ignore_index=True)
    )
    print("Sintese de recorrencia das features mais influentes em B2:")
    display(df_trilhaB_shap_sintese.head(20))
else:
    print("Segmentos SHAP nao executados porque o bloco global de B2 nao foi concluido.")

### Interpretacao da Secao 14 (SHAP de B2)

- O SHAP de B2 abre a caixa preta do estimador nao linear sem alterar a hipotese central de SCAR.
- As importancias globais mostram a hierarquia media das features no score de B2; os segmentos do topo mostram o que pesa especificamente nos casos mais priorizados.
- Mesmo com interpretabilidade mais rica, a leitura continua sendo sobre o score aprendido sob o problema PU observavel, e nao sobre causalidade em `Y`.

## 15) Interpretabilidade linear complementar do B1

Embora o B1 nao seja o vencedor empirico da trilha, ele continua sendo a ancora interpretativa mais simples do PU classico sob SCAR.

Nesta secao, aproveitamos os coeficientes do estimador logistico padronizado para construir uma leitura complementar:
- direcao da associacao com o score;
- contraste com a flexibilidade nao linear revelada pelo SHAP de B2;
- interpretacao cautelosa, sem salto causal.

In [ ]:
# ---------------------------------------------------------
# Interpretabilidade linear complementar do B1
# ---------------------------------------------------------
# Contexto:
# O B1 fornece uma referencia linear, padronizada e mais simples
# de auditar do que o estimador nao linear de B2.
#
# Objetivo estatistico:
# Extrair a direcao dos coeficientes do modelo logistico interno
# do Elkan-Noto, explorando-os como ancora interpretativa.
#
# Interpretacao:
# Como as features numericas passaram por StandardScaler, o sinal
# do coeficiente e a leitura principal. A magnitude pode ser lida
# apenas com cautela, pois continua refletindo a forma linear
# especifica do estimador interno.

b1_preprocess = model_final.named_steps["preprocess"]
b1_estimator = getattr(pu_step_final, "estimator_", getattr(pu_step_final, "estimator", None))

if b1_estimator is None or not hasattr(b1_estimator, "coef_"):
    df_trilhaB_b1_coeficientes = pd.DataFrame(columns=["feature", "coef", "coef_abs", "sinal_associacao", "rank_abs"])
    print("Coeficientes nao disponiveis no estimador interno de B1.")
else:
    b1_feature_names = [str(col) for col in (b1_preprocess.get_feature_names_out() if hasattr(b1_preprocess, "get_feature_names_out") else numeric_feature_cols)]
    coef_values = np.asarray(b1_estimator.coef_, dtype="float64").reshape(-1)
    if len(coef_values) != len(b1_feature_names):
        raise ValueError(f"Inconsistencia entre coeficientes ({len(coef_values)}) e features ({len(b1_feature_names)}).")
    df_trilhaB_b1_coeficientes = pd.DataFrame({"feature": b1_feature_names, "coef": coef_values})
    df_trilhaB_b1_coeficientes["coef_abs"] = df_trilhaB_b1_coeficientes["coef"].abs()
    df_trilhaB_b1_coeficientes["sinal_associacao"] = np.where(df_trilhaB_b1_coeficientes["coef"] >= 0, "positivo", "negativo")
    df_trilhaB_b1_coeficientes["rank_abs"] = df_trilhaB_b1_coeficientes["coef_abs"].rank(method="dense", ascending=False).astype("int64")
    df_trilhaB_b1_coeficientes = df_trilhaB_b1_coeficientes.sort_values(["coef_abs", "feature"], ascending=[False, True]).reset_index(drop=True)
    print("Coeficientes mais positivos de B1:")
    display(df_trilhaB_b1_coeficientes.sort_values(["coef", "feature"], ascending=[False, True]).head(15))
    print("\nCoeficientes mais negativos de B1:")
    display(df_trilhaB_b1_coeficientes.sort_values(["coef", "feature"], ascending=[True, True]).head(15))

### Interpretacao da Secao 15 (coeficientes do B1)

- Os coeficientes de B1 servem como ancora interpretativa linear da trilha.
- O sinal do coeficiente e a leitura mais defensavel: positivo sugere aumento do score do B1; negativo sugere reducao.
- Essa leitura complementa o SHAP de B2: B1 oferece simplicidade linear, enquanto B2 mostra interacoes e nao linearidades.

## 16) Comparacao interna da Trilha B

Antes da persistencia final, consolidamos uma tabela comparativa unica entre B1, B2 e B3.

Essa tabela serve a dois papeis:
- documentar as variantes internas da Trilha B para o artigo/TCC;
- alimentar comparacoes futuras entre trilhas em um schema consistente.

In [ ]:
# ---------------------------------------------------------
# Tabela comparativa interna da Trilha B
# ---------------------------------------------------------
# A tabela organiza desempenho proxy, c estimado, custo e funcao editorial
# de cada componente interno da trilha.

df_trilhaB_compare = pd.DataFrame([
    {
        "codigo_modelo": "B1",
        "model_name": "Elkan-Noto + Logistic Regression",
        "familia_metodologica": "PU_classico_SCAR",
        "base_estimator": "LogisticRegression",
        "componentes_adicionais": "tuning_logistico_controlado",
        "score_principal_componente": "score_pu_raw",
        "auc_proxy_s_val": float(auc_val_final),
        "auc_proxy_s": float(auc_s),
        "recall_at_1pct": metric_at(df_ranking_k, 0.01, "recall_at_k"),
        "recall_at_5pct": metric_at(df_ranking_k, 0.05, "recall_at_k"),
        "recall_at_10pct": metric_at(df_ranking_k, 0.10, "recall_at_k"),
        "precision_at_1pct": metric_at(df_ranking_k, 0.01, "precision_at_k"),
        "precision_at_5pct": metric_at(df_ranking_k, 0.05, "precision_at_k"),
        "precision_at_10pct": metric_at(df_ranking_k, 0.10, "precision_at_k"),
        "c_est": float(c_estimado) if pd.notna(c_estimado) else np.nan,
        "pct_score_raw_gt_1": float((df_scores_test["score_pu_raw"] > 1).mean()) if len(df_scores_test) > 0 else np.nan,
        "pct_score_calibrated_gt_1": np.nan,
        "train_seconds": float(train_seconds_final),
        "converged": bool(converged_final),
        "observacao_interpretativa": "Referencia metodologica principal.",
    },
    {
        "codigo_modelo": "B2",
        "model_name": f"Elkan-Noto + {b2_base_estimator_name}",
        "familia_metodologica": "PU_classico_SCAR",
        "base_estimator": b2_base_estimator_name,
        "componentes_adicionais": "sensibilidade_ao_estimador_base",
        "score_principal_componente": "score_pu_raw",
        "auc_proxy_s_val": float(auc_val_b2),
        "auc_proxy_s": float(auc_test_b2),
        "recall_at_1pct": metric_at(df_ranking_k_b2, 0.01, "recall_at_k"),
        "recall_at_5pct": metric_at(df_ranking_k_b2, 0.05, "recall_at_k"),
        "recall_at_10pct": metric_at(df_ranking_k_b2, 0.10, "recall_at_k"),
        "precision_at_1pct": metric_at(df_ranking_k_b2, 0.01, "precision_at_k"),
        "precision_at_5pct": metric_at(df_ranking_k_b2, 0.05, "precision_at_k"),
        "precision_at_10pct": metric_at(df_ranking_k_b2, 0.10, "precision_at_k"),
        "c_est": float(c_estimado_b2) if pd.notna(c_estimado_b2) else np.nan,
        "pct_score_raw_gt_1": pct_score_raw_gt_1_b2,
        "pct_score_calibrated_gt_1": np.nan,
        "train_seconds": float(train_seconds_b2),
        "converged": bool(b2_converged),
        "observacao_interpretativa": "Sensibilidade ao estimador base sem mudar SCAR.",
    },
    {
        "codigo_modelo": "B3",
        "model_name": f"IsotonicRegression sobre {best_raw_model_code}",
        "familia_metodologica": "PU_classico_SCAR_calibrado",
        "base_estimator": f"{best_raw_base_estimator_name} + IsotonicRegression",
        "componentes_adicionais": "calibracao_monotona_em_validacao",
        "score_principal_componente": "score_pu_calibrated",
        "auc_proxy_s_val": float(auc_val_b3),
        "auc_proxy_s": float(auc_test_b3),
        "recall_at_1pct": metric_at(df_ranking_k_b3, 0.01, "recall_at_k"),
        "recall_at_5pct": metric_at(df_ranking_k_b3, 0.05, "recall_at_k"),
        "recall_at_10pct": metric_at(df_ranking_k_b3, 0.10, "recall_at_k"),
        "precision_at_1pct": metric_at(df_ranking_k_b3, 0.01, "precision_at_k"),
        "precision_at_5pct": metric_at(df_ranking_k_b3, 0.05, "precision_at_k"),
        "precision_at_10pct": metric_at(df_ranking_k_b3, 0.10, "precision_at_k"),
        "c_est": best_raw_c_est,
        "pct_score_raw_gt_1": best_raw_pct_score_gt_1,
        "pct_score_calibrated_gt_1": pct_score_calibrated_gt_1,
        "train_seconds": float(calibration_seconds_b3),
        "converged": True,
        "observacao_interpretativa": "Pos-processamento interpretativo; nao substitui o ranking bruto oficial.",
    },
])

print("Tabela comparativa interna da Trilha B:")
display(df_trilhaB_compare)
print("\nMelhor modelo bruto da trilha:")
print(f"- codigo: {best_raw_model_code}")
print(f"- nome: {best_raw_model_name}")
print("- score oficial para ranking operacional: score_pu_raw")


### Interpretacao da comparacao interna e limites do SCAR

- **B1** e a referencia metodologica principal da trilha e o ponto de partida interpretavel.
- **B2** mede a sensibilidade do score a um unico estimador base nao linear, sem criar zoologico de modelos.
- **B3** acrescenta calibracao monotona ao melhor modelo bruto, mas sem substituir o score oficial de ranking.

**Observacao explicita sobre os limites do SCAR**
- melhorias em B2 e B3 nao eliminam a hipotese SCAR;
- se a selecao dos positivos depender de `x`, a limitacao estrutural permanece;
- a Trilha C existe justamente para reformular o problema para alem do PU classico sob SCAR.


## 17) Salvando resultados e artefatos de reprodutibilidade

Persistimos os artefatos centrais da Trilha B com nomes proprios e sufixo `_smoke` quando `SMOKE=True`.

Conjunto minimo salvo nesta trilha:
- scores de teste de B1, B2 e B3;
- ranking operacional do melhor modelo bruto;
- tabela comparativa interna da Trilha B;
- quadro completo do tuning de B1;
- artefatos novos de interpretabilidade, robustez e diagnostico do score;
- pipeline do melhor modelo bruto, calibrador de B3, feature names, `X_test` transformado e manifest.

In [ ]:
# ---------------------------------------------------------
# Persistencia dos artefatos principais da Trilha B
# ---------------------------------------------------------
# O ranking salvo continua ancorado no score_pu_raw do melhor modelo bruto.

import pickle

artifact_suffix = "_smoke" if SMOKE else ""
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

B1_SCORES_TEST_PATH = OUTPUT_DIR / f"trilhaB_b1_scores_test{artifact_suffix}.parquet"
B2_SCORES_TEST_PATH = OUTPUT_DIR / f"trilhaB_b2_scores_test{artifact_suffix}.parquet"
B3_SCORES_TEST_PATH = OUTPUT_DIR / f"trilhaB_b3_scores_test{artifact_suffix}.parquet"
RANKING_TOP_PATH = OUTPUT_DIR / f"trilhaB_ranking_unlabeled_top100k{artifact_suffix}.parquet"
INTERNAL_COMPARE_PATH = OUTPUT_DIR / f"trilhaB_comparacao_interna{artifact_suffix}.parquet"
B1_TUNING_COMPARE_PATH = OUTPUT_DIR / f"trilhaB_b1_tuning_compare{artifact_suffix}.parquet"
BEST_MODEL_PATH = OUTPUT_DIR / f"trilhaB_best_model_pipeline{artifact_suffix}.pkl"
CALIBRATOR_PATH = OUTPUT_DIR / f"trilhaB_b3_calibrator{artifact_suffix}.pkl"
BEST_FEATURES_PATH = OUTPUT_DIR / f"trilhaB_best_feature_names{artifact_suffix}.parquet"
BEST_XTEST_PATH = OUTPUT_DIR / f"trilhaB_best_X_test_transformado{artifact_suffix}.parquet"
SHAP_GLOBAL_PATH = OUTPUT_DIR / f"trilhaB_shap_global{artifact_suffix}.parquet"
SHAP_TOP_PATH = OUTPUT_DIR / f"trilhaB_shap_top_ranking{artifact_suffix}.parquet"
SHAP_GROUPS_PATH = OUTPUT_DIR / f"trilhaB_shap_grupos{artifact_suffix}.parquet"
SHAP_SINTESIS_PATH = OUTPUT_DIR / f"trilhaB_shap_sintese{artifact_suffix}.parquet"
B2_B3_OVERLAP_PATH = OUTPUT_DIR / f"trilhaB_b2_b3_ranking_overlap{artifact_suffix}.parquet"
HOLDOUT_SENSITIVITY_PATH = OUTPUT_DIR / f"trilhaB_holdout_sensitivity{artifact_suffix}.parquet"
SCORE_GT1_DIAG_PATH = OUTPUT_DIR / f"trilhaB_score_raw_gt1_diagnostico{artifact_suffix}.parquet"
B1_COEF_PATH = OUTPUT_DIR / f"trilhaB_b1_coeficientes{artifact_suffix}.parquet"
MANIFEST_PATH = OUTPUT_DIR / f"trilhaB_artifacts_manifest{artifact_suffix}.parquet"

ranking_unlabeled_topN = best_ranking_unlabeled.head(TOP_N_RANKING).copy()
ranking_unlabeled_topN["codigo_modelo_origem"] = best_raw_model_code

preprocess_best = best_raw_model.named_steps["preprocess"]
X_test_transformado_best = preprocess_best.transform(X_test)
if hasattr(X_test_transformado_best, "toarray"):
    X_test_transformado_best = X_test_transformado_best.toarray()
X_test_transformado_best = np.asarray(X_test_transformado_best, dtype="float32")
transformed_feature_names_best = [
    str(col) for col in (
        preprocess_best.get_feature_names_out()
        if hasattr(preprocess_best, "get_feature_names_out")
        else numeric_feature_cols
    )
]

df_best_X_test_transformado = pd.DataFrame(X_test_transformado_best, columns=transformed_feature_names_best)
df_best_X_test_transformado.insert(0, "s", best_scores_test_df["s"].to_numpy())
df_best_X_test_transformado.insert(0, "ibge", best_scores_test_df["ibge"].astype("string").to_numpy())
df_best_X_test_transformado.insert(0, "id_familia", best_scores_test_df["id_familia"].astype("string").to_numpy())
df_best_X_test_transformado.insert(0, "cpf_agricultor", best_scores_test_df["cpf_agricultor"].astype("string").to_numpy())
df_best_X_test_transformado.insert(0, "indice_linha_original", best_scores_test_df["indice_linha_original"].to_numpy())

df_best_feature_names = pd.DataFrame(
    {
        "ordem_feature": np.arange(len(transformed_feature_names_best), dtype="int32"),
        "feature": transformed_feature_names_best,
        "feature_origem": transformed_feature_names_best,
    }
)

with open(BEST_MODEL_PATH, "wb") as f_out:
    pickle.dump(best_raw_model, f_out, protocol=pickle.HIGHEST_PROTOCOL)
with open(CALIBRATOR_PATH, "wb") as f_out:
    pickle.dump(calibrator_b3, f_out, protocol=pickle.HIGHEST_PROTOCOL)

def relpath_str(path_ref: Path) -> str:
    return str(path_ref.relative_to(ROOT)) if path_ref.is_relative_to(ROOT) else str(path_ref)

df_manifest = pd.DataFrame([{
    "artifact_schema_version": "3.0",
    "trilha": "B",
    "notebook_origem": "notebooks/21_trilha_B_pu_classico.ipynb",
    "notebook_base": "notebooks/07_trilhaA_baseline_pu_elkanoto.ipynb",
    "smoke": bool(SMOKE),
    "smoke_max_rows": int(SMOKE_MAX_ROWS),
    "smoke_rows_original": int(smoke_info["n_rows_original"]),
    "smoke_rows_final": int(smoke_info["n_rows_final"]),
    "score_oficial": "score_pu_raw",
    "score_auxiliar": "score_pu_clipped",
    "score_calibrado": "score_pu_calibrated",
    "selection_rule_b1": best_selection_rule,
    "selection_rule_b3": f"prefer_B2_if_auc_val_gain_gt_{BEST_RAW_MODEL_TOL:.0e}_else_B1",
    "data_path": relpath_str(DATA_PATH),
    "output_dir": relpath_str(OUTPUT_DIR),
    "seed": int(SEED),
    "group_col": str(GROUP_COL),
    "hold_out_ratio_main": float(HOLD_OUT_RATIO_MAIN),
    "hold_out_ratio_grid": ",".join([f"{x:.2f}" for x in HOLD_OUT_RATIO_GRID]),
    "b2_base_estimator": b2_base_estimator_name,
    "best_raw_model_code": best_raw_model_code,
    "best_raw_model_name": best_raw_model_name,
    "best_raw_base_estimator": best_raw_base_estimator_name,
    "best_raw_auc_val": float(best_auc_val_raw),
    "best_raw_auc_test": float(best_auc_test_raw),
    "best_raw_train_seconds": float(best_raw_train_seconds),
    "best_raw_converged": bool(best_raw_converged),
    "best_raw_c_est": best_raw_c_est,
    "b3_auc_val": float(auc_val_b3),
    "b3_auc_test": float(auc_test_b3),
    "b3_calibration_seconds": float(calibration_seconds_b3),
    "shap_status_b2": str(globals().get("shap_status_b2", "nao_executado")),
    "shap_engine_b2": str(globals().get("shap_engine_b2", "na")),
    "python_version": sys.version.split()[0],
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
    "pulearn_version": pulearn.__version__,
    "b1_scores_file": B1_SCORES_TEST_PATH.name,
    "b2_scores_file": B2_SCORES_TEST_PATH.name,
    "b3_scores_file": B3_SCORES_TEST_PATH.name,
    "ranking_file": RANKING_TOP_PATH.name,
    "internal_compare_file": INTERNAL_COMPARE_PATH.name,
    "b1_tuning_compare_file": B1_TUNING_COMPARE_PATH.name,
    "best_model_file": BEST_MODEL_PATH.name,
    "calibrator_file": CALIBRATOR_PATH.name,
    "best_features_file": BEST_FEATURES_PATH.name,
    "best_xtest_file": BEST_XTEST_PATH.name,
    "shap_global_file": SHAP_GLOBAL_PATH.name,
    "shap_top_file": SHAP_TOP_PATH.name,
    "shap_groups_file": SHAP_GROUPS_PATH.name,
    "shap_sintese_file": SHAP_SINTESIS_PATH.name,
    "b2_b3_overlap_file": B2_B3_OVERLAP_PATH.name,
    "holdout_sensitivity_file": HOLDOUT_SENSITIVITY_PATH.name,
    "score_gt1_diag_file": SCORE_GT1_DIAG_PATH.name,
    "b1_coef_file": B1_COEF_PATH.name,
    "manifest_file": MANIFEST_PATH.name,
}])

df_scores_test.to_parquet(B1_SCORES_TEST_PATH, index=False)
df_scores_test_b2.to_parquet(B2_SCORES_TEST_PATH, index=False)
df_scores_test_b3.to_parquet(B3_SCORES_TEST_PATH, index=False)
ranking_unlabeled_topN.to_parquet(RANKING_TOP_PATH, index=False)
df_trilhaB_compare.to_parquet(INTERNAL_COMPARE_PATH, index=False)
df_model_compare_final.to_parquet(B1_TUNING_COMPARE_PATH, index=False)
df_best_feature_names.to_parquet(BEST_FEATURES_PATH, index=False)
df_best_X_test_transformado.to_parquet(BEST_XTEST_PATH, index=False)
df_trilhaB_shap_global.to_parquet(SHAP_GLOBAL_PATH, index=False)
df_trilhaB_shap_top.to_parquet(SHAP_TOP_PATH, index=False)
df_trilhaB_shap_groups.to_parquet(SHAP_GROUPS_PATH, index=False)
df_trilhaB_shap_sintese.to_parquet(SHAP_SINTESIS_PATH, index=False)
df_trilhaB_b2_b3_ranking_overlap.to_parquet(B2_B3_OVERLAP_PATH, index=False)
df_holdout_sensitivity.to_parquet(HOLDOUT_SENSITIVITY_PATH, index=False)
df_trilhaB_score_raw_gt1_diag.to_parquet(SCORE_GT1_DIAG_PATH, index=False)
df_trilhaB_b1_coeficientes.to_parquet(B1_COEF_PATH, index=False)
df_manifest.to_parquet(MANIFEST_PATH, index=False)

print("Arquivos salvos com sucesso:")
for path_saved in [
    B1_SCORES_TEST_PATH,
    B2_SCORES_TEST_PATH,
    B3_SCORES_TEST_PATH,
    RANKING_TOP_PATH,
    INTERNAL_COMPARE_PATH,
    B1_TUNING_COMPARE_PATH,
    BEST_MODEL_PATH,
    CALIBRATOR_PATH,
    BEST_FEATURES_PATH,
    BEST_XTEST_PATH,
    SHAP_GLOBAL_PATH,
    SHAP_TOP_PATH,
    SHAP_GROUPS_PATH,
    SHAP_SINTESIS_PATH,
    B2_B3_OVERLAP_PATH,
    HOLDOUT_SENSITIVITY_PATH,
    SCORE_GT1_DIAG_PATH,
    B1_COEF_PATH,
    MANIFEST_PATH,
]:
    print(f"- {path_saved}")

### Interpretacao da Secao 17 (persistencia de artefatos)

- ficam salvos scores completos de B1, B2 e B3, o ranking operacional do melhor modelo bruto, a tabela comparativa interna, o tuning de B1 e os artefatos tecnicos de reproducao;
- o ranking salvo continua ancorado em `score_pu_raw` do melhor modelo bruto;
- o score calibrado salvo em B3 serve como camada interpretativa adicional, nao como substituto automatico do ranking oficial.


## 18) Conclusoes e fechamento tecnico da Trilha B

Esta secao sintetiza o que a Trilha B agora entrega de forma auditavel:
- baseline linear interpretavel (B1);
- sensibilidade a estimador base nao linear (B2);
- calibracao monotona do melhor modelo bruto (B3);
- interpretabilidade nao linear via SHAP;
- robustez do ranking frente a calibracao e a `hold_out_ratio`;
- diagnostico especifico dos casos com `score_pu_raw > 1`.

In [ ]:
# ---------------------------------------------------------
# Encerramento executivo da Trilha B
# ---------------------------------------------------------
# Contexto:
# Esta celula resume os achados metodologicos apos incluir
# interpretabilidade, estabilidade e sensibilidade estrutural.
#
# Interpretacao:
# O fechamento privilegia o que muda a leitura do score e o que
# permanece como limite estrutural da hipotese SCAR.

print("Resumo executivo da Trilha B:")
print("- B1: referencia metodologica principal com Elkan-Noto + Logistic Regression")
print(f"- B2: sensibilidade ao estimador base com {b2_base_estimator_name}")
print(f"- B3: calibracao monotona aplicada sobre {best_raw_model_code}")
print(f"- Melhor modelo bruto interno: {best_raw_model_code} ({best_raw_model_name})")
print("- Score oficial para ranking operacional: score_pu_raw do melhor modelo bruto")
print("- Score calibrado: uso interpretativo complementar")

if "df_overlap_b3_detalhado" in globals() and not df_overlap_b3_detalhado.empty:
    min_overlap = float(df_overlap_b3_detalhado["overlap"].min())
    print(f"- Sobreposicao minima entre B2 bruto e B3 calibrado nos recortes avaliados: {min_overlap:.4f}")

if "df_holdout_sensitivity" in globals() and not df_holdout_sensitivity.empty:
    auc_span = float(df_holdout_sensitivity["auc_proxy_s"].max() - df_holdout_sensitivity["auc_proxy_s"].min())
    print(f"- Variacao de AUC_s na grade de hold_out_ratio: {auc_span:.6f}")

if "df_trilhaB_shap_sintese" in globals() and not df_trilhaB_shap_sintese.empty:
    top_features = ", ".join(df_trilhaB_shap_sintese.head(5)["feature"].astype(str).tolist())
    print(f"- Features recorrentes no SHAP de B2: {top_features}")

if "df_score_raw_gt1_profile" in globals() and not df_score_raw_gt1_profile.empty:
    pct_gt1 = df_score_raw_gt1_profile.loc[
        df_score_raw_gt1_profile["grupo_score_raw"] == "score_pu_raw_gt_1",
        "pct_casos_teste",
    ]
    if not pct_gt1.empty:
        print(f"- Peso dos casos com score_pu_raw > 1 na base de teste: {float(pct_gt1.iloc[0]):.4%}")

### Leitura consolidada do fechamento

A Trilha B passa a responder de forma mais completa a pergunta: **como corrigimos o vies de selecao assumindo SCAR?**

Fechamento consolidado:
- **B1** permanece como referencia metodologica principal e ancora interpretativa linear;
- **B2** mede a sensibilidade do score a um unico estimador base nao linear, sem mudar a hipotese central;
- **B3** calibra o melhor modelo bruto para melhorar a leitura interpretativa do score;
- o **score oficial** da trilha para ranking operacional permanece sendo `score_pu_raw` do melhor modelo bruto entre B1 e B2;
- o **score calibrado** e salvo e documentado, mas nao substitui automaticamente o ranking oficial;
- o bloco de SHAP amplia a interpretabilidade de B2 sem prometer leitura causal em `Y`;
- a limitacao estrutural do SCAR permanece e justifica a transicao posterior para a Trilha C.

## 19) Insight exploratorio de prevalencia latente sob SCAR

In [ ]:
# ---------------------------------------------------------
# Analise complementar: prevalencia latente sob hipotese SCAR
# ---------------------------------------------------------
# Esta leitura depende inteiramente de SCAR e nao compete com o produto
# principal do notebook, que continua sendo o ranking operacional bruto.

N = len(df)
Ns1 = int((df["s"] == 1).sum())
ps1 = Ns1 / N

if pd.notna(best_raw_c_est) and best_raw_c_est > 0:
    pi_hat = ps1 / best_raw_c_est
    Ny1_hat = Ns1 / best_raw_c_est
    missing_hat = Ny1_hat - Ns1

    print("Insight complementar (exploratorio sob SCAR):")
    print(f"- modelo bruto de referencia: {best_raw_model_code} ({best_raw_model_name})")
    print(f"- Total de observacoes (N): {N:,}")
    print(f"- Positivos observados (s=1): {Ns1:,}")
    print(f"- P(s=1): {ps1:.4f}")
    print(f"- c estimado do modelo de referencia: {best_raw_c_est:.4f}")
    print(f"- pi_hat = P(y=1) aprox {pi_hat:.4f}")
    print(f"- Ny1_hat aprox {Ny1_hat:,.0f}")
    print(f"- Missing positives aprox {missing_hat:,.0f}")
    if pi_hat > 1:
        print("- Aviso: pi_hat > 1 sugere violacao plausivel de SCAR ou c instavel.")
else:
    print("c do melhor modelo bruto indisponivel ou invalido; nao e possivel calcular o insight sob SCAR.")


Como analise complementar, a estimativa de prevalencia latente sob hipotese SCAR sugere massa potencial de positivos alem dos beneficiarios observados. Ainda assim, esse resultado nao e o produto principal da Trilha B: o foco do notebook permanece no ranking operacional baseado em `score_pu_raw` do melhor modelo bruto. Por depender diretamente de SCAR, `pi_hat` deve ser lido apenas como diagnostico exploratorio e nao como medida direta de elegibilidade real.
